In [2]:
!pip install -q transformers datasets accelerate

In [3]:


from __future__ import annotations

import argparse
import ast
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEFAULT_SYSTEM_PROMPT = "You are an expert Python competitive programmer."


@dataclass
class Task:
    task_id: str
    prompt: str
    test: str
    entry_point: str


@dataclass
class TokenTrace:
    position: int
    token_id: int
    token_text: str
    entropy: float
    margin: float
    top1_id: int
    top1_text: str
    top2_id: int
    top2_text: str


@dataclass
class Generation:
    token_ids: list[int]
    code: str
    trace: list[TokenTrace]
    latency_s: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-name", default=DEFAULT_MODEL)
    parser.add_argument("--output-dir", default="/kaggle/working/top2_counterfactual_pilot")
    parser.add_argument("--task-ids", default="", help="Comma-separated HumanEval task IDs.")
    parser.add_argument("--num-tasks", type=int, default=10, help="Used only when --task-ids is empty.")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--candidates-per-task", type=int, default=3)
    parser.add_argument("--controls-per-task", type=int, default=3)
    parser.add_argument("--edge-buffer", type=int, default=5)
    parser.add_argument("--timeout-s", type=int, default=8)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--trust-remote-code", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    # Jupyter/Kaggle executes a cell with an internal ``-f <kernel.json>``
    # argument. Accept that one argument pair while keeping normal CLI typos
    # visible to the user.
    args, unknown = parser.parse_known_args()
    if unknown:
        if len(unknown) == 2 and unknown[0] == "-f":
            return args
        parser.error(f"unrecognized arguments: {' '.join(unknown)}")
    return args


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_tasks(task_ids: str, num_tasks: int) -> list[Task]:
    dataset = load_dataset("openai_humaneval", split="test")
    requested_ids = {item.strip() for item in task_ids.split(",") if item.strip()}
    tasks = [
        Task(
            task_id=row["task_id"],
            prompt=row["prompt"],
            test=row["test"],
            entry_point=row["entry_point"],
        )
        for row in dataset
        if not requested_ids or row["task_id"] in requested_ids
    ]
    if requested_ids:
        missing = requested_ids - {task.task_id for task in tasks}
        if missing:
            raise ValueError(f"Unknown HumanEval task IDs: {sorted(missing)}")
        return tasks
    return tasks[:num_tasks]


def build_prompt(task: Task, tokenizer: Any) -> str:
    user_prompt = (
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not use Markdown. Do not explain.\n\n"
        f"{task.prompt}"
    )
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{DEFAULT_SYSTEM_PROMPT}\n\n{user_prompt}"


def strip_markdown_fences(text: str) -> str:
    text = (text or "").strip()
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    blocks = [block.strip() for block in blocks if block.strip()]
    if blocks:
        keywords = ("def ", "import ", "from ", "class ", "return ", "assert ")
        return max(blocks, key=lambda block: sum(key in block for key in keywords) * 10 + len(block))
    return text.replace("```python", "").replace("```py", "").replace("```", "").strip()


def extract_code(raw_output: str, entry_point: str) -> str:
    text = strip_markdown_fences(raw_output)
    for marker in ("Explanation:", "Example:", "Examples:", "# Explanation"):
        marker_index = text.find(marker)
        if marker_index != -1:
            text = text[:marker_index].strip()
    match = re.search(rf"def\s+{re.escape(entry_point)}\s*\(", text)
    if match:
        imports = [
            line.strip()
            for line in text[: match.start()].splitlines()
            if line.strip().startswith(("import ", "from "))
        ]
        function_code = text[match.start() :].strip()
        return "\n".join(imports + ([""] if imports else []) + [function_code]).strip()
    return text.strip()


def evaluate(task: Task, raw_output: str, timeout_s: int) -> tuple[bool, str | None]:
    code = extract_code(raw_output, task.entry_point)
    try:
        ast.parse(code)
    except SyntaxError as error:
        return False, f"SyntaxError: {error.msg} at line {error.lineno}"

    prelude = (
        "from typing import *\nimport math\nimport re\nimport itertools\n"
        "import collections\nimport functools\nimport heapq\nimport bisect\n"
        "import string\nimport statistics\nfrom collections import *\n\n"
    )
    source = prelude + code + "\n\n" + task.test + f"\n\ncheck({task.entry_point})\n"
    with tempfile.TemporaryDirectory() as temp_dir:
        candidate_path = Path(temp_dir) / "candidate.py"
        candidate_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"Timeout: exceeded {timeout_s}s"
    if result.returncode == 0:
        return True, None
    stderr = (result.stderr or result.stdout or "unknown execution failure").strip()
    return False, stderr[-800:]


def model_input_device(model: Any) -> torch.device:
    return model.get_input_embeddings().weight.device


def entropy_and_top2(logits: torch.Tensor, tokenizer: Any, position: int) -> TokenTrace:
    logits = logits.float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    entropy = float((-(probabilities * log_probabilities).sum()).item())
    top_values, top_ids = torch.topk(logits, k=2, dim=-1)
    top1_id, top2_id = int(top_ids[0].item()), int(top_ids[1].item())
    return TokenTrace(
        position=position,
        token_id=top1_id,
        token_text=tokenizer.decode([top1_id]),
        entropy=entropy,
        margin=float((top_values[0] - top_values[1]).item()),
        top1_id=top1_id,
        top1_text=tokenizer.decode([top1_id]),
        top2_id=top2_id,
        top2_text=tokenizer.decode([top2_id]),
    )


@torch.inference_mode()
def greedy_generate(model: Any, tokenizer: Any, prompt: str, max_new_tokens: int) -> Generation:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    started = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    generated_ids: list[int] = []
    trace: list[TokenTrace] = []
    eos_token_id = tokenizer.eos_token_id

    for position in range(max_new_tokens):
        token_trace = entropy_and_top2(next_logits[0], tokenizer, position)
        trace.append(token_trace)
        next_token_id = token_trace.top1_id
        generated_ids.append(next_token_id)
        if eos_token_id is not None and next_token_id == eos_token_id:
            break
        next_token = torch.tensor([[next_token_id]], device=input_device, dtype=torch.long)
        outputs = model(next_token, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=trace,
        latency_s=time.perf_counter() - started,
    )


@torch.inference_mode()
def generate_top1_top2_batch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    top1_id: int,
    top2_id: int,
    max_new_tokens: int,
) -> tuple[Generation, Generation]:
    """Continue top-1 and top-2 branches in a batch of two equally long prefixes."""

    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=input_device).unsqueeze(0)
    shared_prefix = torch.cat([prompt_ids, prefix_tensor], dim=1) if prefix_ids else prompt_ids
    candidates = torch.tensor([[top1_id], [top2_id]], dtype=torch.long, device=input_device)
    input_ids = torch.cat([shared_prefix.repeat(2, 1), candidates], dim=1)
    started = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    branch_ids = [[top1_id], [top2_id]]
    branch_trace: list[list[TokenTrace]] = [[], []]
    finished = [False, False]
    eos_token_id = tokenizer.eos_token_id
    remaining = max(max_new_tokens - len(prefix_ids) - 1, 0)

    for step in range(remaining):
        next_ids: list[int] = []
        for branch_index in range(2):
            token_trace = entropy_and_top2(next_logits[branch_index], tokenizer, len(prefix_ids) + 1 + step)
            branch_trace[branch_index].append(token_trace)
            token_id = eos_token_id if finished[branch_index] and eos_token_id is not None else token_trace.top1_id
            branch_ids[branch_index].append(int(token_id))
            if eos_token_id is not None and token_id == eos_token_id:
                finished[branch_index] = True
            next_ids.append(int(token_id))
        if all(finished):
            break
        next_tensor = torch.tensor(next_ids, dtype=torch.long, device=input_device).unsqueeze(1)
        outputs = model(next_tensor, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    latency_s = time.perf_counter() - started
    complete_ids = [prefix_ids + branch for branch in branch_ids]
    return (
        Generation(
            token_ids=complete_ids[0],
            code=tokenizer.decode(complete_ids[0], skip_special_tokens=True),
            trace=branch_trace[0],
            latency_s=latency_s,
        ),
        Generation(
            token_ids=complete_ids[1],
            code=tokenizer.decode(complete_ids[1], skip_special_tokens=True),
            trace=branch_trace[1],
            latency_s=latency_s,
        ),
    )


def select_positions(
    trace: list[TokenTrace],
    candidates_per_task: int,
    controls_per_task: int,
    edge_buffer: int,
    rng: np.random.Generator,
) -> list[tuple[str, TokenTrace]]:
    valid = trace[edge_buffer : max(len(trace) - edge_buffer, edge_buffer)]
    high_entropy = sorted(valid, key=lambda item: item.entropy, reverse=True)[:candidates_per_task]
    high_positions = {item.position for item in high_entropy}
    controls_pool = [item for item in valid if item.position not in high_positions]
    controls_count = min(controls_per_task, len(controls_pool))
    controls = (
        [controls_pool[index] for index in rng.choice(len(controls_pool), size=controls_count, replace=False)]
        if controls_count
        else []
    )
    return [("high_entropy", item) for item in high_entropy] + [("random_control", item) for item in controls]


def baseline_record(task: Task, generation: Generation, passed: bool, error: str | None) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": passed,
        "error": error,
        "code": generation.code,
        "token_ids": generation.token_ids,
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def write_summary(branch_records: list[dict[str, Any]], output_dir: Path) -> None:
    rows = []
    for selection_type in ("high_entropy", "random_control"):
        group = [record for record in branch_records if record["selection_type"] == selection_type]
        if not group:
            continue
        recoverable = sum(bool(record["recoverable"]) for record in group)
        rows.append(
            {
                "selection_type": selection_type,
                "n_positions": len(group),
                "recoverable": recoverable,
                "recovery_rate": recoverable / len(group),
                "mean_entropy": float(np.mean([record["entropy"] for record in group])),
                "mean_extra_tokens": float(np.mean([record["extra_tokens"] for record in group])),
            }
        )
    report = {"rows": rows, "created_at_unix": time.time()}
    (output_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    markdown = ["# Top-1 vs Top-2 counterfactual pilot", "", "| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |", "|---|---:|---:|---:|---:|---:|"]
    markdown.extend(
        "| {selection_type} | {n_positions} | {recoverable} | {recovery_rate:.1%} | {mean_entropy:.3f} | {mean_extra_tokens:.1f} |".format(**row)
        for row in rows
    )
    (output_dir / "summary.md").write_text("\n".join(markdown) + "\n", encoding="utf-8")
    print("\n".join(markdown))


def main() -> None:
    args = parse_args()
def run_notebook(
    *,
    task_ids: str,
    model_name: str = DEFAULT_MODEL,
    output_dir: str = "/kaggle/working/top2_counterfactual_pilot",
    max_new_tokens: int = 256,
    candidates_per_task: int = 3,
    controls_per_task: int = 3,
    edge_buffer: int = 5,
    timeout_s: int = 8,
    seed: int = 42,
    trust_remote_code: bool = False,
    overwrite: bool = False,
) -> None:
    """Run the pilot directly from a Kaggle notebook cell.

    Example:
        run_notebook(task_ids="HumanEval/26,HumanEval/38", candidates_per_task=2)
    """

    args = argparse.Namespace(
        model_name=model_name,
        output_dir=output_dir,
        task_ids=task_ids,
        num_tasks=10,
        max_new_tokens=max_new_tokens,
        candidates_per_task=candidates_per_task,
        controls_per_task=controls_per_task,
        edge_buffer=edge_buffer,
        timeout_s=timeout_s,
        seed=seed,
        trust_remote_code=trust_remote_code,
        overwrite=overwrite,
    )
    main(args)


def main(args: argparse.Namespace | None = None) -> None:
    args = args or parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This pilot requires a Kaggle GPU session.")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = output_dir / "baselines.jsonl"
    branch_path = output_dir / "branches.jsonl"
    metadata_path = output_dir / "metadata.json"
    if args.overwrite:
        for path in (baseline_path, branch_path, metadata_path):
            if path.exists():
                path.unlink()

    tasks = load_tasks(args.task_ids, args.num_tasks)
    metadata_path.write_text(
        json.dumps({"args": vars(args), "tasks": [task.task_id for task in tasks]}, indent=2),
        encoding="utf-8",
    )
    print(f"Loading one model across {torch.cuda.device_count()} GPU(s): {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=args.trust_remote_code)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=args.trust_remote_code,
    )
    model.eval()

    existing_baselines = {record["task_id"]: record for record in read_jsonl(baseline_path)}
    existing_branches = {record["task_id"] for record in read_jsonl(branch_path)}
    for task in tasks:
        if task.task_id not in existing_baselines:
            print(f"Baseline {task.task_id}")
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(model, tokenizer, prompt, args.max_new_tokens)
            passed, error = evaluate(task, generation.code, args.timeout_s)
            record = baseline_record(task, generation, passed, error)
            append_jsonl(baseline_path, record)
            existing_baselines[task.task_id] = record
            print(f"  {'PASS' if passed else 'FAIL'} | {len(generation.token_ids)} tokens")

        baseline = existing_baselines[task.task_id]
        if baseline["passed"]:
            print(f"Skip {task.task_id}: baseline passed (pilot targets failures).")
            continue
        if task.task_id in existing_branches:
            print(f"Skip {task.task_id}: branches already saved.")
            continue

        prompt = build_prompt(task, tokenizer)
        trace = [TokenTrace(**item) for item in baseline["trace"]]
        selected = select_positions(
            trace,
            candidates_per_task=args.candidates_per_task,
            controls_per_task=args.controls_per_task,
            edge_buffer=args.edge_buffer,
            rng=rng,
        )
        if not selected:
            print(f"Skip {task.task_id}: too few generated tokens for valid branch positions.")
            continue
        print(f"Branching {task.task_id}: {len(selected)} positions")
        for selection_type, point in selected:
            prefix_ids = baseline["token_ids"][: point.position]
            top1_generation, top2_generation = generate_top1_top2_batch(
                model,
                tokenizer,
                prompt,
                prefix_ids=prefix_ids,
                top1_id=point.top1_id,
                top2_id=point.top2_id,
                max_new_tokens=args.max_new_tokens,
            )
            top1_passed, top1_error = evaluate(task, top1_generation.code, args.timeout_s)
            top2_passed, top2_error = evaluate(task, top2_generation.code, args.timeout_s)
            record = {
                "task_id": task.task_id,
                "selection_type": selection_type,
                "position": point.position,
                "position_relative": point.position / max(len(baseline["token_ids"]) - 1, 1),
                "entropy": point.entropy,
                "margin": point.margin,
                "top1_token": point.top1_text,
                "top2_token": point.top2_text,
                "top1_passed": top1_passed,
                "top1_error": top1_error,
                "top2_passed": top2_passed,
                "top2_error": top2_error,
                "recoverable": (not top1_passed) and top2_passed,
                "top1_matches_baseline": top1_generation.code == baseline["code"],
                "extra_tokens": len(top1_generation.token_ids) + len(top2_generation.token_ids) - 2 * len(prefix_ids),
                "branch_latency_s": top1_generation.latency_s,
                "top1_code": top1_generation.code,
                "top2_code": top2_generation.code,
            }
            append_jsonl(branch_path, record)
            print(
                f"  {selection_type} t={point.position:>3} H={point.entropy:.3f} "
                f"top1={'P' if top1_passed else 'F'} top2={'P' if top2_passed else 'F'}"
            )
        existing_branches.add(task.task_id)
        write_summary(read_jsonl(branch_path), output_dir)

    write_summary(read_jsonl(branch_path), output_dir)
    print(f"\nSaved resumable outputs to: {output_dir}")


if __name__ == "__main__":
    main()

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skip HumanEval/0: baseline passed (pilot targets failures).
Skip HumanEval/1: baseline passed (pilot targets failures).
Skip HumanEval/2: baseline passed (pilot targets failures).
Skip HumanEval/3: baseline passed (pilot targets failures).
Skip HumanEval/4: baseline passed (pilot targets failures).
Skip HumanEval/5: baseline passed (pilot targets failures).
Skip HumanEval/6: baseline passed (pilot targets failures).
Skip HumanEval/7: baseline passed (pilot targets failures).
Skip HumanEval/8: baseline passed (pilot targets failures).
Skip HumanEval/9: baseline passed (pilot targets failures).
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [6]:
run_notebook(
    task_ids="HumanEval/26",
    candidates_per_task=1,
    controls_per_task=1,
    max_new_tokens=256,
)

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Baseline HumanEval/26
  FAIL | 53 tokens
Branching HumanEval/26: 2 positions
  high_entropy t= 18 H=0.891 top1=F top2=F
  random_control t=  8 H=0.000 top1=F top2=F
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [2]:
"""
This pilot enumerates every eligible token position of HumanEval/26, forces
the model's top-2 token, completes greedily, evaluates the result, and saves
resumable JSONL/CSV outputs.
"""

import ast
import csv
import gc
import json
import keyword
import shutil
import time
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


PILOT_TASK_ID = "HumanEval/26"
PILOT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PILOT_OUTPUT_DIR = Path("/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b")
PILOT_MAX_NEW_TOKENS = 256
PILOT_EDGE_BUFFER = 2
PILOT_TIMEOUT_S = 8


def classify_token(token_text: str) -> str:
    """Coarse lexical category used only for post-hoc interpretation."""
    if token_text == "":
        return "empty"
    if "\n" in token_text:
        return "newline"
    if token_text.isspace():
        return "whitespace"
    stripped = token_text.strip()
    if keyword.iskeyword(stripped):
        return "keyword"
    if stripped.isidentifier():
        return "identifier"
    if stripped in {
        "+", "-", "*", "/", "//", "%", "**", "=", "==", "!=", "<", ">",
        "<=", ">=", ":=", "&", "|", "^", "~", "<<", ">>",
    }:
        return "operator"
    try:
        ast.literal_eval(stripped)
        return "literal"
    except Exception:
        return "other"


def code_ast_signature(raw_output: str, entry_point: str) -> dict[str, Any]:
    """Return a small AST signature without treating it as a selection signal."""
    code = extract_code(raw_output, entry_point)
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return {
            "parses": False,
            "node_count": 0,
            "node_types": [],
        }
    node_types = [type(node).__name__ for node in ast.walk(tree)]
    return {
        "parses": True,
        "node_count": len(node_types),
        "node_types": sorted(set(node_types)),
    }


def ast_distances(
    baseline_signature: dict[str, Any],
    branch_signature: dict[str, Any],
) -> tuple[float, float]:
    """Jaccard type distance and normalized AST-size distance."""
    baseline_types = set(baseline_signature["node_types"])
    branch_types = set(branch_signature["node_types"])
    union = baseline_types | branch_types
    type_distance = (
        1.0 - len(baseline_types & branch_types) / len(union)
        if union
        else 0.0
    )
    baseline_size = baseline_signature["node_count"]
    branch_size = branch_signature["node_count"]
    size_distance = abs(baseline_size - branch_size) / max(
        baseline_size,
        branch_size,
        1,
    )
    return type_distance, size_distance


@torch.inference_mode()
def generate_forced_token_branch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
    max_new_tokens: int,
) -> Generation:
    """Force one token after a greedy prefix, then continue greedily.

    Only the alternative branch is generated. The corresponding top-1 branch
    is the already-saved deterministic greedy baseline.
    """
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    generated_ids = list(prefix_ids) + [int(forced_token_id)]
    generated_tensor = torch.tensor(
        [generated_ids],
        dtype=torch.long,
        device=input_device,
    )
    input_ids = torch.cat([prompt_ids, generated_tensor], dim=1)
    eos_token_id = tokenizer.eos_token_id

    started = time.perf_counter()
    if eos_token_id is None or forced_token_id != eos_token_id:
        outputs = model(input_ids, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]
        remaining = max(max_new_tokens - len(generated_ids), 0)

        for _ in range(remaining):
            next_token_id = int(torch.argmax(next_logits[0]).item())
            generated_ids.append(next_token_id)
            if eos_token_id is not None and next_token_id == eos_token_id:
                break
            next_token = torch.tensor(
                [[next_token_id]],
                dtype=torch.long,
                device=input_device,
            )
            outputs = model(next_token, past_key_values=cache, use_cache=True)
            cache = outputs.past_key_values
            next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=[],
        latency_s=time.perf_counter() - started,
    )


def write_exhaustive_summary(
    records: list[dict[str, Any]],
    output_dir: Path,
    verbose: bool = False,
) -> None:
    records = sorted(records, key=lambda row: row["position"])
    summary_fields = [
        "position",
        "position_relative",
        "entropy",
        "margin",
        "top1_share_within_top2",
        "top2_share_within_top2",
        "top1_token",
        "top2_token",
        "top1_class",
        "top2_class",
        "semantic_class_change",
        "top2_passed",
        "top2_parses",
        "ast_type_distance",
        "ast_size_distance",
        "branch_tokens",
        "branch_latency_s",
        "error",
    ]
    with (output_dir / "exhaustive_summary.csv").open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=summary_fields)
        writer.writeheader()
        for record in records:
            writer.writerow({field: record.get(field) for field in summary_fields})

    recoveries = [record for record in records if record["top2_passed"]]
    by_entropy = sorted(records, key=lambda row: row["entropy"], reverse=True)
    entropy_rank = {
        record["position"]: rank
        for rank, record in enumerate(by_entropy, start=1)
    }
    report = {
        "task_id": PILOT_TASK_ID,
        "model": PILOT_MODEL,
        "evaluated_positions": len(records),
        "recoveries": len(recoveries),
        "recovery_positions": [record["position"] for record in recoveries],
        "recovery_entropy_ranks": [
            entropy_rank[record["position"]] for record in recoveries
        ],
        "highest_entropy_positions": [
            {
                "position": record["position"],
                "entropy": record["entropy"],
                "top1_token": record["top1_token"],
                "top2_token": record["top2_token"],
                "top2_passed": record["top2_passed"],
            }
            for record in by_entropy[:10]
        ],
    }
    (output_dir / "decision_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    if verbose:
        print("\n=== DECISION REPORT ===")
        print(json.dumps(report, indent=2, ensure_ascii=False))


def run_exhaustive_pilot(overwrite: bool = False) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator in Kaggle before running.")

    torch.manual_seed(42)
    PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    baseline_path = PILOT_OUTPUT_DIR / "baseline.json"
    branches_path = PILOT_OUTPUT_DIR / "exhaustive_branches.jsonl"

    if overwrite:
        for path in (
            baseline_path,
            branches_path,
            PILOT_OUTPUT_DIR / "exhaustive_summary.csv",
            PILOT_OUTPUT_DIR / "decision_report.json",
        ):
            if path.exists():
                path.unlink()

    tasks = load_tasks(PILOT_TASK_ID, num_tasks=1)
    if len(tasks) != 1 or tasks[0].task_id != PILOT_TASK_ID:
        raise RuntimeError(f"Could not load exactly {PILOT_TASK_ID}.")
    task = tasks[0]

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {PILOT_MODEL} on {torch.cuda.device_count()} GPU(s)...")
    tokenizer = AutoTokenizer.from_pretrained(PILOT_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        PILOT_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    prompt = build_prompt(task, tokenizer)

    if baseline_path.exists():
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        print("Loaded saved baseline.")
    else:
        print(f"Generating greedy baseline for {PILOT_TASK_ID}...")
        baseline_generation = greedy_generate(
            model,
            tokenizer,
            prompt,
            PILOT_MAX_NEW_TOKENS,
        )
        baseline_passed, baseline_error = evaluate(
            task,
            baseline_generation.code,
            PILOT_TIMEOUT_S,
        )
        baseline = baseline_record(
            task,
            baseline_generation,
            baseline_passed,
            baseline_error,
        )
        baseline_path.write_text(
            json.dumps(baseline, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    if baseline["passed"]:
        raise RuntimeError(
            "The greedy baseline passed, so recoverability is undefined for "
            "this run. Stop and send the baseline.json file back."
        )

    trace = [TokenTrace(**item) for item in baseline["trace"]]
    baseline_signature = code_ast_signature(baseline["code"], task.entry_point)
    special_ids = set(tokenizer.all_special_ids)
    eligible_points = [
        point
        for point in trace[
            PILOT_EDGE_BUFFER : max(
                len(trace) - PILOT_EDGE_BUFFER,
                PILOT_EDGE_BUFFER,
            )
        ]
        if point.top1_id not in special_ids and point.top2_id not in special_ids
    ]
    print(
        f"Baseline FAIL with {len(baseline['token_ids'])} tokens. "
        f"Eligible positions: {len(eligible_points)}"
    )

    existing_records = read_jsonl(branches_path)
    completed_positions = {record["position"] for record in existing_records}

    for index, point in enumerate(eligible_points, start=1):
        if point.position in completed_positions:
            continue

        prefix_ids = baseline["token_ids"][: point.position]
        branch = generate_forced_token_branch(
            model,
            tokenizer,
            prompt,
            prefix_ids,
            point.top2_id,
            PILOT_MAX_NEW_TOKENS,
        )
        passed, error = evaluate(task, branch.code, PILOT_TIMEOUT_S)
        branch_signature = code_ast_signature(branch.code, task.entry_point)
        ast_type_distance, ast_size_distance = ast_distances(
            baseline_signature,
            branch_signature,
        )
        probability_ratio = float(torch.exp(torch.tensor(-point.margin)).item())
        top1_share_within_top2 = 1.0 / (1.0 + probability_ratio)
        top2_share_within_top2 = probability_ratio / (1.0 + probability_ratio)
        top1_class = classify_token(point.top1_text)
        top2_class = classify_token(point.top2_text)

        record = {
            "task_id": task.task_id,
            "model": PILOT_MODEL,
            "position": point.position,
            "position_relative": point.position
            / max(len(baseline["token_ids"]) - 1, 1),
            "entropy": point.entropy,
            "margin": point.margin,
            # Shares renormalized over top-1 and top-2 only, not full-vocab probabilities.
            "top1_share_within_top2": top1_share_within_top2,
            "top2_share_within_top2": top2_share_within_top2,
            "top1_token_id": point.top1_id,
            "top2_token_id": point.top2_id,
            "top1_token": point.top1_text,
            "top2_token": point.top2_text,
            "top1_class": top1_class,
            "top2_class": top2_class,
            "semantic_class_change": top1_class != top2_class,
            "top2_passed": passed,
            "top2_parses": branch_signature["parses"],
            "ast_type_distance": ast_type_distance,
            "ast_size_distance": ast_size_distance,
            "branch_tokens": len(branch.token_ids),
            "branch_latency_s": branch.latency_s,
            "error": error,
            "top2_code": branch.code,
        }
        append_jsonl(branches_path, record)
        existing_records.append(record)
        completed_positions.add(point.position)
        print(
            f"[{index:>3}/{len(eligible_points)}] "
            f"t={point.position:>3} H={point.entropy:>6.3f} "
            f"{point.top1_text!r} -> {point.top2_text!r} "
            f"{'PASS' if passed else 'FAIL'}"
        )

        # Keep the summary usable even if Kaggle interrupts the session.
        write_exhaustive_summary(existing_records, PILOT_OUTPUT_DIR, verbose=False)

    write_exhaustive_summary(existing_records, PILOT_OUTPUT_DIR, verbose=True)
    archive_path = shutil.make_archive(
        str(PILOT_OUTPUT_DIR),
        "zip",
        root_dir=PILOT_OUTPUT_DIR,
    )
    print("\nDownload this ZIP from Kaggle Output and send it back:")
    print(archive_path)


# First run: preserves and resumes partial output if Kaggle disconnects.
run_exhaustive_pilot(overwrite=False)


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading Qwen/Qwen2.5-Coder-7B-Instruct on 2 GPU(s)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generating greedy baseline for HumanEval/26...
Baseline FAIL with 53 tokens. Eligible positions: 49
[  1/49] t=  2 H= 0.000 ' import' -> ' List' FAIL
[  2/49] t=  3 H= 0.422 ' List' -> ' *\n' FAIL
[  3/49] t=  4 H= 0.200 '\n\n\n' -> '\n' PASS
[  4/49] t=  5 H= 0.002 'def' -> 'from' PASS
[  5/49] t=  6 H= 0.000 ' remove' -> 'remove' FAIL
[  6/49] t=  7 H= 0.000 '_duplicates' -> ' duplicates' FAIL
[  7/49] t=  8 H= 0.000 '(numbers' -> '(nums' FAIL
[  8/49] t=  9 H= 0.000 ':' -> ':\n' FAIL
[  9/49] t= 10 H= 0.000 ' List' -> 'List' FAIL
[ 10/49] t= 11 H= 0.000 '[int' -> '(int' FAIL
[ 11/49] t= 12 H= 0.000 '])' -> ']' FAIL
[ 12/49] t= 13 H= 0.000 ' ->' -> ' -' FAIL
[ 13/49] t= 14 H= 0.000 ' List' -> 'List' FAIL
[ 14/49] t= 15 H= 0.000 '[int' -> '(int' FAIL
[ 15/49] t= 16 H= 0.002 ']:\n' -> ']:' FAIL
[ 16/49] t= 17 H= 0.002 '   ' -> '    \n' FAIL
[ 17/49] t= 18 H= 0.891 ' seen' -> ' """' FAIL
[ 18/49] t= 19 H= 0.008 ' =' -> '_once' FAIL
[ 19/49] t= 20 H= 0.050 ' set' -> ' {}\n' FAIL
[ 20/49]

In [3]:
"""Pilot: entropy vs. true top-1/top-2 probability margin on HumanEval/26.

Paste this file into a NEW Kaggle cell after running:
  1. the dependency-installation cell; and
  2. the large definitions cell from notebook0cef4050e3.ipynb.

This does NOT regenerate the 49 branches. It loads their saved outcomes and
performs one teacher-forced model pass to recover the true full-vocabulary
probabilities p1 and p2 at every baseline position.

Before running, make the previous ZIP available in either /kaggle/working or
/kaggle/input. The code finds it recursively.
"""

import csv
import gc
import json
import math
import shutil
import zipfile
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


MARGIN_TASK_ID = "HumanEval/26"
MARGIN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
SOURCE_ARCHIVE_NAME = "exhaustive_branch_humaneval_26_qwen25_7b.zip"
MARGIN_OUTPUT_DIR = Path("/kaggle/working/margin_pilot_humaneval_26")
MARGIN_EXTRACT_DIR = Path("/kaggle/working/margin_pilot_source")
MARGIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MARGIN_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def find_source_artifacts() -> Path:
    """Use the prior output directory or safely extract its ZIP."""
    direct_dir = Path(
        "/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b"
    )
    required = {
        "baseline.json",
        "exhaustive_branches.jsonl",
        "exhaustive_summary.csv",
        "decision_report.json",
    }
    if direct_dir.is_dir() and required.issubset(
        {path.name for path in direct_dir.iterdir()}
    ):
        return direct_dir

    candidates = []
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            candidates.extend(root.rglob(SOURCE_ARCHIVE_NAME))
    if not candidates:
        raise FileNotFoundError(
            f"Upload {SOURCE_ARCHIVE_NAME} to Kaggle Input or Working first."
        )

    archive_path = sorted(candidates, key=lambda path: len(str(path)))[0]
    print(f"Using source archive: {archive_path}")
    with zipfile.ZipFile(archive_path) as archive:
        names = {Path(name).name for name in archive.namelist()}
        if not required.issubset(names):
            raise RuntimeError(
                f"Source ZIP is missing: {sorted(required - names)}"
            )
        for info in archive.infolist():
            name = Path(info.filename)
            if name.is_absolute() or ".." in name.parts:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")
        archive.extractall(MARGIN_EXTRACT_DIR)
    return MARGIN_EXTRACT_DIR


def read_jsonl_file(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def binary_ranking_metrics(
    rows: list[dict[str, Any]],
    score: Callable[[dict[str, Any]], float],
    descending: bool,
) -> dict[str, Any]:
    """Compute descriptive ranking metrics without sklearn."""
    ranked = sorted(rows, key=score, reverse=descending)
    labels = [int(row["top2_passed"]) for row in ranked]
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        raise RuntimeError("Both positive and negative outcomes are required.")

    average_precision = sum(
        sum(labels[: index + 1]) / (index + 1)
        for index, label in enumerate(labels)
        if label
    ) / positives

    positive_scores = [
        score(row) for row in rows if row["top2_passed"]
    ]
    negative_scores = [
        score(row) for row in rows if not row["top2_passed"]
    ]
    pairwise_wins = 0.0
    for positive in positive_scores:
        for negative in negative_scores:
            if positive == negative:
                pairwise_wins += 0.5
            elif (positive > negative) == descending:
                pairwise_wins += 1.0
    auroc = pairwise_wins / (positives * negatives)

    ranks = {
        row["position"]: index
        for index, row in enumerate(ranked, start=1)
    }
    return {
        "auroc": auroc,
        "average_precision": average_precision,
        "positive_ranks": {
            str(row["position"]): ranks[row["position"]]
            for row in rows
            if row["top2_passed"]
        },
        "top_k": {
            str(k): {
                "positions": [row["position"] for row in ranked[:k]],
                "recoveries": sum(labels[:k]),
                "recall": sum(labels[:k]) / positives,
                "precision": sum(labels[:k]) / k,
            }
            for k in (1, 3, 5, 10, 15)
        },
    }


def ordinal_ranks(values: list[float], descending: bool) -> list[int]:
    """Deterministic ordinal ranks; ties keep position order."""
    order = sorted(
        range(len(values)),
        key=lambda index: values[index],
        reverse=descending,
    )
    ranks = [0] * len(values)
    for rank, index in enumerate(order):
        ranks[index] = rank
    return ranks


@torch.inference_mode()
def recompute_full_vocabulary_probabilities(
    model: Any,
    tokenizer: Any,
    prompt: str,
    baseline_token_ids: list[int],
) -> list[dict[str, Any]]:
    """Teacher-force the baseline and recover p1/p2 for every generated token."""
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    baseline_ids = torch.tensor(
        [baseline_token_ids],
        dtype=torch.long,
        device=input_device,
    )
    full_ids = torch.cat([prompt_ids, baseline_ids], dim=1)
    prompt_length = prompt_ids.shape[1]

    outputs = model(full_ids, use_cache=False)
    # Index prompt_length - 1 predicts generated token 0.
    predicted_logits = outputs.logits[
        0,
        prompt_length - 1 : prompt_length - 1 + len(baseline_token_ids),
        :,
    ].float()
    log_probabilities = torch.log_softmax(predicted_logits, dim=-1)
    probabilities = log_probabilities.exp()
    top_probabilities, top_ids = torch.topk(probabilities, k=2, dim=-1)
    entropies = -(probabilities * log_probabilities).sum(dim=-1)
    top_logits = torch.gather(predicted_logits, 1, top_ids)

    results = []
    for position in range(len(baseline_token_ids)):
        p1 = float(top_probabilities[position, 0].item())
        p2 = float(top_probabilities[position, 1].item())
        z1 = float(top_logits[position, 0].item())
        z2 = float(top_logits[position, 1].item())
        results.append(
            {
                "position": position,
                "recomputed_top1_id": int(top_ids[position, 0].item()),
                "recomputed_top2_id": int(top_ids[position, 1].item()),
                "p1": p1,
                "p2": p2,
                "probability_margin": p1 - p2,
                "relative_probability_margin": (p1 - p2) / max(p1, 1e-30),
                "top2_to_top1_ratio": p2 / max(p1, 1e-30),
                "top2_mass": p1 + p2,
                "recomputed_logit_margin": z1 - z2,
                "recomputed_entropy": float(entropies[position].item()),
            }
        )

    del outputs, predicted_logits, log_probabilities, probabilities
    torch.cuda.empty_cache()
    return results


def make_scatter(rows: list[dict[str, Any]], output_path: Path) -> bool:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib unavailable; skipping scatter plot.")
        return False

    colors = [
        "#198754" if row["top2_passed"] else "#9aa0a6"
        for row in rows
    ]
    plt.figure(figsize=(8, 5.5))
    plt.scatter(
        [row["probability_margin"] for row in rows],
        [row["recomputed_entropy"] for row in rows],
        c=colors,
        alpha=0.85,
    )
    for row in rows:
        if row["top2_passed"]:
            plt.annotate(
                str(row["position"]),
                (row["probability_margin"], row["recomputed_entropy"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=9,
                fontweight="bold",
            )
    plt.xlabel("True probability margin p1 - p2 (smaller = more ambiguous)")
    plt.ylabel("Token entropy (larger = more uncertain)")
    plt.title("HumanEval/26: uncertainty signals and top-2 recoveries")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return True


def run_margin_pilot() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator.")

    source_dir = find_source_artifacts()
    baseline = json.loads(
        (source_dir / "baseline.json").read_text(encoding="utf-8")
    )
    branch_rows = read_jsonl_file(
        source_dir / "exhaustive_branches.jsonl"
    )
    if baseline["task_id"] != MARGIN_TASK_ID:
        raise RuntimeError(f"Unexpected task: {baseline['task_id']}")
    if len(branch_rows) != 49 or len(
        {row["position"] for row in branch_rows}
    ) != 49:
        raise RuntimeError("Expected 49 unique exhaustive branch records.")

    task = load_tasks(MARGIN_TASK_ID, num_tasks=1)[0]

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {MARGIN_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(MARGIN_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MARGIN_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    prompt = build_prompt(task, tokenizer)

    recomputed = recompute_full_vocabulary_probabilities(
        model,
        tokenizer,
        prompt,
        baseline["token_ids"],
    )
    by_position = {row["position"]: row for row in recomputed}
    trace_by_position = {
        row["position"]: row for row in baseline["trace"]
    }

    joined = []
    mismatch_rows = []
    for branch in sorted(branch_rows, key=lambda row: row["position"]):
        position = branch["position"]
        probability_row = by_position[position]
        trace_row = trace_by_position[position]
        mismatch = {
            "position": position,
            "stored_top1": trace_row["top1_id"],
            "new_top1": probability_row["recomputed_top1_id"],
            "stored_top2": trace_row["top2_id"],
            "new_top2": probability_row["recomputed_top2_id"],
        }
        if (
            mismatch["stored_top1"] != mismatch["new_top1"]
            or mismatch["stored_top2"] != mismatch["new_top2"]
        ):
            mismatch_rows.append(mismatch)

        joined.append(
            {
                **branch,
                **probability_row,
                "stored_entropy": branch["entropy"],
                "stored_logit_margin": branch["margin"],
                "entropy_abs_error": abs(
                    branch["entropy"]
                    - probability_row["recomputed_entropy"]
                ),
                "logit_margin_abs_error": abs(
                    branch["margin"]
                    - probability_row["recomputed_logit_margin"]
                ),
            }
        )

    if mismatch_rows:
        print(
            f"WARNING: {len(mismatch_rows)} top-token mismatches after reload."
        )
        print(json.dumps(mismatch_rows[:10], indent=2))

    entropy_values = [row["recomputed_entropy"] for row in joined]
    probability_margins = [row["probability_margin"] for row in joined]
    entropy_ranks = ordinal_ranks(entropy_values, descending=True)
    margin_ranks = ordinal_ranks(probability_margins, descending=False)
    denominator = max(len(joined) - 1, 1)
    for index, row in enumerate(joined):
        row["entropy_rank_fraction"] = entropy_ranks[index] / denominator
        row["margin_rank_fraction"] = margin_ranks[index] / denominator
        # Fixed label-free combination: equal average of the two ranks.
        row["combined_rank_uncertainty"] = 1.0 - 0.5 * (
            row["entropy_rank_fraction"] + row["margin_rank_fraction"]
        )

    metrics = {
        "entropy_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["recomputed_entropy"],
            descending=True,
        ),
        "probability_margin_low": binary_ranking_metrics(
            joined,
            score=lambda row: row["probability_margin"],
            descending=False,
        ),
        "top2_probability_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["p2"],
            descending=True,
        ),
        "top2_to_top1_ratio_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["top2_to_top1_ratio"],
            descending=True,
        ),
        "combined_rank_entropy_margin": binary_ranking_metrics(
            joined,
            score=lambda row: row["combined_rank_uncertainty"],
            descending=True,
        ),
    }

    correlation = float(
        np.corrcoef(entropy_values, probability_margins)[0, 1]
    )
    positive_details = [
        {
            "position": row["position"],
            "entropy": row["recomputed_entropy"],
            "p1": row["p1"],
            "p2": row["p2"],
            "probability_margin": row["probability_margin"],
            "top2_to_top1_ratio": row["top2_to_top1_ratio"],
            "top1_token": row["top1_token"],
            "top2_token": row["top2_token"],
        }
        for row in joined
        if row["top2_passed"]
    ]
    report = {
        "task_id": MARGIN_TASK_ID,
        "model": MARGIN_MODEL,
        "positions": len(joined),
        "recoveries": sum(row["top2_passed"] for row in joined),
        "top_token_mismatches_after_reload": len(mismatch_rows),
        "max_entropy_recompute_error": max(
            row["entropy_abs_error"] for row in joined
        ),
        "max_logit_margin_recompute_error": max(
            row["logit_margin_abs_error"] for row in joined
        ),
        "pearson_entropy_vs_probability_margin": correlation,
        "positive_details": positive_details,
        "ranking_metrics": metrics,
        "interpretation_rule": {
            "margin": "smaller p1-p2 means greater top-2 ambiguity",
            "entropy": "larger entropy means greater full-distribution uncertainty",
            "combined": "fixed equal-weight average of within-problem ranks; no labels used",
        },
    }

    report_path = MARGIN_OUTPUT_DIR / "margin_pilot_report.json"
    csv_path = MARGIN_OUTPUT_DIR / "margin_positions.csv"
    plot_path = MARGIN_OUTPUT_DIR / "margin_entropy_scatter.png"
    report_path.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    excluded_from_csv = {"top2_code", "error"}
    csv_fields = sorted(
        {
            key
            for row in joined
            for key in row
            if key not in excluded_from_csv
        }
    )
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=csv_fields)
        writer.writeheader()
        for row in joined:
            writer.writerow({key: row.get(key) for key in csv_fields})

    plot_created = make_scatter(joined, plot_path)
    archive_path = shutil.make_archive(
        str(MARGIN_OUTPUT_DIR),
        "zip",
        root_dir=MARGIN_OUTPUT_DIR,
    )

    print("\n=== MARGIN PILOT REPORT ===")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print("\nDownload and send back:")
    print(archive_path)
    if plot_created:
        print(f"Scatter plot: {plot_path}")


run_margin_pilot()


Loading Qwen/Qwen2.5-Coder-7B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[
  {
    "position": 28,
    "stored_top1": 1372,
    "new_top1": 1629,
    "stored_top2": 1629,
    "new_top2": 1372
  }
]

=== MARGIN PILOT REPORT ===
{
  "task_id": "HumanEval/26",
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "positions": 49,
  "recoveries": 3,
  "top_token_mismatches_after_reload": 1,
  "max_entropy_recompute_error": 0.00312197208404541,
  "max_logit_margin_recompute_error": 0.03125,
  "pearson_entropy_vs_probability_margin": -0.9547017301820563,
  "positive_details": [
    {
      "position": 4,
      "entropy": 0.19993337988853455,
      "p1": 0.9598875045776367,
      "p2": 0.025580130517482758,
      "probability_margin": 0.934307374060154,
      "top2_to_top1_ratio": 0.02664909210245252,
      "top1_token": "\n\n\n",
      "top2_token": "\n"
    },
    {
      "position": 5,
      "entropy": 0.0022935837041586637,
      "p1": 0.9997833371162415,
      "p2": 0.00014201307203620672,
      "probability_margin": 0.9996413240442052,
      "top2_to_top1_ratio": 0

In [1]:
"""Semantic-lookahead pilot for the exhaustive HumanEval/26 experiment.

Paste this complete file into a NEW Kaggle cell after running:
  1. the dependency-installation cell; and
  2. the large definitions cell from notebook0cef4050e3.ipynb.

Also add ``exhaustive_branch_humaneval_26_qwen25_7b.zip`` as a Kaggle
input (or leave its extracted output under /kaggle/working).

The experiment does not use tests to select positions. At every previously
evaluated position it forces the exact saved alternative token, rolls out only
12 tokens, and ranks the branch using pre-registered, label-free trajectory
features. The saved pass/fail outcome is used only after ranking for evaluation.
"""

import ast
import csv
import gc
import json
import keyword
import math
import re
import shutil
import time
import zipfile
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


LOOKAHEAD_TASK_ID = "HumanEval/26"
LOOKAHEAD_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
SOURCE_ARCHIVE_NAME = "exhaustive_branch_humaneval_26_qwen25_7b.zip"
LOOKAHEAD_TOKENS = 12
OUTPUT_DIR = Path("/kaggle/working/semantic_lookahead_pilot_humaneval_26")
EXTRACT_DIR = Path("/kaggle/working/semantic_lookahead_source")
OUTPUT_ZIP = Path(
    "/kaggle/working/semantic_lookahead_pilot_humaneval_26.zip"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def find_source_artifacts() -> Path:
    """Find the exhaustive pilot or safely extract its ZIP."""
    direct = Path(
        "/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b"
    )
    required = {"baseline.json", "exhaustive_branches.jsonl"}
    if direct.is_dir() and required.issubset(
        {path.name for path in direct.iterdir()}
    ):
        return direct

    candidates = []
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            candidates.extend(root.rglob(SOURCE_ARCHIVE_NAME))
    if not candidates:
        raise FileNotFoundError(
            f"Add {SOURCE_ARCHIVE_NAME} to Kaggle Input first."
        )

    archive_path = sorted(candidates, key=lambda path: len(str(path)))[0]
    print(f"Using source archive: {archive_path}")
    with zipfile.ZipFile(archive_path) as archive:
        names = {Path(name).name for name in archive.namelist()}
        if not required.issubset(names):
            raise RuntimeError(
                f"Source ZIP is missing: {sorted(required - names)}"
            )
        for info in archive.infolist():
            path = Path(info.filename)
            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")
        archive.extractall(EXTRACT_DIR)

    for candidate in (EXTRACT_DIR, *EXTRACT_DIR.iterdir()):
        if candidate.is_dir() and required.issubset(
            {path.name for path in candidate.iterdir()}
        ):
            return candidate
    raise RuntimeError("Could not locate extracted source artifacts.")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()


def token_class(text: str) -> str:
    """Coarse class for a code token; independent of test outcomes."""
    if not text:
        return "empty"
    if text.isspace() or "\n" in text:
        return "whitespace"
    stripped = text.strip()
    if keyword.iskeyword(stripped):
        return "keyword"
    if stripped.isidentifier():
        return "identifier"
    if stripped in {
        "+", "-", "*", "/", "//", "%", "**", "=", "==", "!=", "<", ">",
        "<=", ">=", ":=", "&", "|", "^", "~", "<<", ">>", "->",
    }:
        return "operator"
    try:
        ast.literal_eval(stripped)
        return "literal"
    except Exception:
        return "other"


def levenshtein_distance(left: list[int], right: list[int]) -> int:
    """Memory-efficient edit distance."""
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right) + 1))
    for i, left_value in enumerate(left, start=1):
        current = [i]
        for j, right_value in enumerate(right, start=1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (left_value != right_value),
                )
            )
        previous = current
    return previous[-1]


def aligned_mismatch(left: list[int], right: list[int], start: int = 0) -> float:
    length = max(len(left), len(right))
    if length <= start:
        return 0.0
    mismatches = 0
    for index in range(start, length):
        left_value = left[index] if index < len(left) else None
        right_value = right[index] if index < len(right) else None
        mismatches += left_value != right_value
    return mismatches / (length - start)


def weighted_token_divergence(
    baseline_texts: list[str],
    branch_texts: list[str],
) -> float:
    """Discount formatting and simple renames, emphasize code decisions."""
    length = max(len(baseline_texts), len(branch_texts))
    if length == 0:
        return 0.0
    total = 0.0
    for index in range(length):
        left = baseline_texts[index] if index < len(baseline_texts) else ""
        right = branch_texts[index] if index < len(branch_texts) else ""
        if left == right:
            continue
        left_class = token_class(left)
        right_class = token_class(right)
        classes = {left_class, right_class}
        if classes == {"whitespace"}:
            weight = 0.0
        elif classes == {"identifier"}:
            weight = 0.20
        elif "whitespace" in classes:
            weight = 0.25
        elif classes & {"keyword", "operator", "literal"}:
            weight = 1.0
        else:
            weight = 0.50
        total += weight
    return total / length


ANCHOR_OPERATOR_PATTERN = re.compile(
    r"==|!=|<=|>=|:=|//|\*\*|->|[+\-*/%<>=&|^~]"
)
ANCHOR_CALL_PATTERN = re.compile(
    r"\b([A-Za-z_]\w*)\s*\("
)
ANCHOR_METHOD_PATTERN = re.compile(
    r"\.\s*([A-Za-z_]\w*)\b"
)
ANCHOR_NUMBER_PATTERN = re.compile(
    r"(?<![\w.])(?:0[xob][0-9A-Fa-f]+|\d+(?:\.\d+)?)(?![\w.])"
)
ANCHOR_WORD_PATTERN = re.compile(r"\b[A-Za-z_]\w*\b")


def code_anchors(text: str) -> list[str]:
    """Extract intent-bearing lexical anchors while ignoring plain names."""
    anchors = []
    for word in ANCHOR_WORD_PATTERN.findall(text):
        if keyword.iskeyword(word):
            anchors.append(f"keyword:{word}")
    anchors.extend(
        f"operator:{value}" for value in ANCHOR_OPERATOR_PATTERN.findall(text)
    )
    anchors.extend(
        f"call:{value}" for value in ANCHOR_CALL_PATTERN.findall(text)
        if not keyword.iskeyword(value)
    )
    anchors.extend(
        f"method:{value}" for value in ANCHOR_METHOD_PATTERN.findall(text)
    )
    anchors.extend(
        f"number:{value}" for value in ANCHOR_NUMBER_PATTERN.findall(text)
    )
    return anchors


def anchor_features(baseline_text: str, branch_text: str) -> dict[str, float]:
    baseline = set(code_anchors(baseline_text))
    branch = set(code_anchors(branch_text))
    union = baseline | branch
    distance = 1.0 - len(baseline & branch) / len(union) if union else 0.0

    baseline_lexemes = max(len(ANCHOR_WORD_PATTERN.findall(baseline_text)), 1)
    branch_lexemes = max(len(ANCHOR_WORD_PATTERN.findall(branch_text)), 1)
    baseline_density = len(code_anchors(baseline_text)) / baseline_lexemes
    branch_density = len(code_anchors(branch_text)) / branch_lexemes

    # A prose/docstring detour can be very different but has low code density.
    density_retention = min(
        (branch_density + 0.05) / (baseline_density + 0.05),
        1.0,
    )
    return {
        "anchor_distance": distance,
        "baseline_anchor_density": baseline_density,
        "branch_anchor_density": branch_density,
        "anchor_score": distance * density_retention,
    }


@torch.inference_mode()
def generate_short_forced_rollout(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
    rollout_tokens: int,
) -> dict[str, Any]:
    """Measure branch probabilities and generate a short greedy continuation."""
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    if prefix_ids:
        prefix_tensor = torch.tensor(
            [prefix_ids], dtype=torch.long, device=input_device
        )
        context_ids = torch.cat([prompt_ids, prefix_tensor], dim=1)
    else:
        context_ids = prompt_ids

    started = time.perf_counter()
    outputs = model(context_ids, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[0, -1, :].float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    top_probabilities, top_ids = torch.topk(probabilities, k=2)
    entropy = float(-(probabilities * log_probabilities).sum().item())
    forced_probability = float(probabilities[forced_token_id].item())
    forced_log_probability = float(log_probabilities[forced_token_id].item())
    forced_rank = int((logits > logits[forced_token_id]).sum().item()) + 1

    generated = [int(forced_token_id)]
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None or forced_token_id != eos_token_id:
        next_input = torch.tensor(
            [[forced_token_id]], dtype=torch.long, device=input_device
        )
        outputs = model(next_input, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[0, -1, :]

        for _ in range(max(rollout_tokens - 1, 0)):
            next_id = int(torch.argmax(next_logits).item())
            generated.append(next_id)
            if eos_token_id is not None and next_id == eos_token_id:
                break
            next_input = torch.tensor(
                [[next_id]], dtype=torch.long, device=input_device
            )
            outputs = model(
                next_input,
                past_key_values=cache,
                use_cache=True,
            )
            cache = outputs.past_key_values
            next_logits = outputs.logits[0, -1, :]

    result = {
        "branch_token_ids": generated,
        "branch_token_texts": [
            tokenizer.decode([token_id], skip_special_tokens=False)
            for token_id in generated
        ],
        "same_run_top1_id": int(top_ids[0].item()),
        "same_run_top2_id": int(top_ids[1].item()),
        "same_run_p1": float(top_probabilities[0].item()),
        "same_run_p2": float(top_probabilities[1].item()),
        "same_run_probability_margin": float(
            top_probabilities[0].item() - top_probabilities[1].item()
        ),
        "same_run_entropy": entropy,
        "forced_probability": forced_probability,
        "forced_log_probability": forced_log_probability,
        "forced_rank": forced_rank,
        "latency_s": time.perf_counter() - started,
    }
    del outputs, logits, log_probabilities, probabilities
    return result


def ordinal_percentile_scores(
    rows: list[dict[str, Any]],
    field: str,
    descending: bool = True,
) -> dict[int, float]:
    """Map values to [0,1]; 1 is selected first. Ties keep token position."""
    ordered = sorted(
        rows,
        key=lambda row: (
            -float(row[field]) if descending else float(row[field]),
            int(row["position"]),
        ),
    )
    denominator = max(len(ordered) - 1, 1)
    return {
        int(row["position"]): 1.0 - index / denominator
        for index, row in enumerate(ordered)
    }


def ranking_metrics(
    rows: list[dict[str, Any]],
    field: str,
    descending: bool,
) -> dict[str, Any]:
    ranked = sorted(
        rows,
        key=lambda row: float(row[field]),
        reverse=descending,
    )
    labels = [int(row["top2_passed"]) for row in ranked]
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        raise RuntimeError("Ranking evaluation needs both outcome classes.")

    average_precision = sum(
        sum(labels[:index + 1]) / (index + 1)
        for index, label in enumerate(labels)
        if label
    ) / positives
    positive_values = [
        float(row[field]) for row in rows if row["top2_passed"]
    ]
    negative_values = [
        float(row[field]) for row in rows if not row["top2_passed"]
    ]
    wins = 0.0
    for positive in positive_values:
        for negative in negative_values:
            if positive == negative:
                wins += 0.5
            elif (positive > negative) == descending:
                wins += 1.0

    rank_by_position = {
        int(row["position"]): rank
        for rank, row in enumerate(ranked, start=1)
    }
    return {
        "auroc": wins / (positives * negatives),
        "average_precision": average_precision,
        "positive_ranks": {
            str(row["position"]): rank_by_position[int(row["position"])]
            for row in rows
            if row["top2_passed"]
        },
        "top_k": {
            str(k): {
                "positions": [
                    int(row["position"]) for row in ranked[:k]
                ],
                "recoveries": sum(labels[:k]),
                "recall": sum(labels[:k]) / positives,
                "precision": sum(labels[:k]) / k,
            }
            for k in (1, 3, 5, 10, 15)
        },
    }


def make_plot(rows: list[dict[str, Any]], output_path: Path) -> bool:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return False

    methods = [
        ("entropy", "same_run_entropy", True),
        ("margin", "same_run_probability_margin", False),
        ("edit", "normalized_edit_distance", True),
        ("semantic", "semantic_lookahead_score", True),
    ]
    cutoffs = [1, 3, 5, 10, 15]
    values = []
    for label, field, descending in methods:
        ranked = sorted(
            rows,
            key=lambda row: float(row[field]),
            reverse=descending,
        )
        values.append(
            [
                sum(int(row["top2_passed"]) for row in ranked[:k])
                for k in cutoffs
            ]
        )

    x = np.arange(len(cutoffs))
    width = 0.19
    plt.figure(figsize=(9, 5.2))
    for index, ((label, _, _), counts) in enumerate(zip(methods, values)):
        plt.bar(x + (index - 1.5) * width, counts, width, label=label)
    plt.xticks(x, [f"top-{k}" for k in cutoffs])
    plt.yticks([0, 1, 2, 3])
    plt.ylabel("Recoveries found (out of 3)")
    plt.title("HumanEval/26: selection under a branch budget")
    plt.legend()
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return True


def write_csv(rows: list[dict[str, Any]], path: Path) -> None:
    fields = [
        "position",
        "top1_token",
        "top2_token",
        "top2_passed",
        "same_run_top1_id",
        "same_run_top2_id",
        "forced_rank",
        "same_run_p1",
        "same_run_p2",
        "same_run_probability_margin",
        "same_run_entropy",
        "forced_probability",
        "normalized_edit_distance",
        "aligned_mismatch",
        "persistence_after_forced",
        "weighted_semantic_divergence",
        "anchor_distance",
        "baseline_anchor_density",
        "branch_anchor_density",
        "anchor_score",
        "semantic_lookahead_score",
        "saved_pair_matches_same_run",
        "short_rollout_matches_saved_branch",
        "baseline_snippet",
        "branch_snippet",
    ]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in sorted(rows, key=lambda item: item["position"]):
            writer.writerow({field: row.get(field) for field in fields})


def run_semantic_lookahead_pilot(overwrite: bool = False) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator.")

    source_dir = find_source_artifacts()
    baseline = json.loads(
        (source_dir / "baseline.json").read_text(encoding="utf-8")
    )
    branches = sorted(
        read_jsonl(source_dir / "exhaustive_branches.jsonl"),
        key=lambda row: int(row["position"]),
    )
    if len(branches) != 49 or len({row["position"] for row in branches}) != 49:
        raise RuntimeError("Expected the 49 unique exhaustive interventions.")
    if baseline["passed"]:
        raise RuntimeError("This pilot expects the saved failing baseline.")

    tasks = load_tasks(LOOKAHEAD_TASK_ID, num_tasks=1)
    if len(tasks) != 1 or tasks[0].task_id != LOOKAHEAD_TASK_ID:
        raise RuntimeError(f"Could not load exactly {LOOKAHEAD_TASK_ID}.")

    rollout_path = OUTPUT_DIR / "short_rollouts.jsonl"
    if overwrite and rollout_path.exists():
        rollout_path.unlink()
    completed = {
        int(row["position"]): row
        for row in read_jsonl(rollout_path)
    } if rollout_path.exists() else {}

    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(42)
    print(f"Loading {LOOKAHEAD_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(LOOKAHEAD_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompt = build_prompt(tasks[0], tokenizer)
    model = AutoModelForCausalLM.from_pretrained(
        LOOKAHEAD_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    baseline_ids = [int(value) for value in baseline["token_ids"]]
    branch_by_position = {
        int(row["position"]): row for row in branches
    }
    for index, position in enumerate(sorted(branch_by_position), start=1):
        if position in completed:
            print(f"[{index:2d}/49] t={position:2d} cached")
            continue
        source = branch_by_position[position]
        forced_id = int(source["top2_token_id"])
        rollout_length = min(
            LOOKAHEAD_TOKENS,
            max(len(baseline_ids) - position, 1),
        )
        generated = generate_short_forced_rollout(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            prefix_ids=baseline_ids[:position],
            forced_token_id=forced_id,
            rollout_tokens=rollout_length,
        )

        baseline_window = baseline_ids[position:position + rollout_length]
        branch_window = generated["branch_token_ids"]
        baseline_texts = [
            tokenizer.decode([token_id], skip_special_tokens=False)
            for token_id in baseline_window
        ]
        baseline_snippet = tokenizer.decode(
            baseline_window, skip_special_tokens=True
        )
        branch_snippet = tokenizer.decode(
            branch_window, skip_special_tokens=True
        )
        anchors = anchor_features(baseline_snippet, branch_snippet)
        saved_pair = {
            int(source["top1_token_id"]),
            int(source["top2_token_id"]),
        }
        same_run_pair = {
            generated["same_run_top1_id"],
            generated["same_run_top2_id"],
        }

        candidate_prefix = tokenizer.decode(
            baseline_ids[:position] + branch_window,
            skip_special_tokens=True,
        )
        saved_code = str(source.get("top2_code", ""))
        record = {
            "task_id": LOOKAHEAD_TASK_ID,
            "model": LOOKAHEAD_MODEL,
            "lookahead_tokens": rollout_length,
            "position": position,
            "top1_token": source["top1_token"],
            "top2_token": source["top2_token"],
            "top1_token_id": int(source["top1_token_id"]),
            "top2_token_id": forced_id,
            "top2_passed": bool(source["top2_passed"]),
            "baseline_token_ids": baseline_window,
            "baseline_token_texts": baseline_texts,
            "baseline_snippet": baseline_snippet,
            "branch_snippet": branch_snippet,
            "normalized_edit_distance": levenshtein_distance(
                baseline_window, branch_window
            ) / max(len(baseline_window), len(branch_window), 1),
            "aligned_mismatch": aligned_mismatch(
                baseline_window, branch_window
            ),
            "persistence_after_forced": aligned_mismatch(
                baseline_window, branch_window, start=1
            ),
            "weighted_semantic_divergence": weighted_token_divergence(
                baseline_texts,
                generated["branch_token_texts"],
            ),
            **anchors,
            **generated,
            "saved_pair_matches_same_run": saved_pair == same_run_pair,
            "short_rollout_matches_saved_branch": (
                saved_code.startswith(candidate_prefix)
            ),
        }
        append_jsonl(rollout_path, record)
        completed[position] = record
        print(
            f"[{index:2d}/49] t={position:2d} "
            f"edit={record['normalized_edit_distance']:.3f} "
            f"persist={record['persistence_after_forced']:.3f} "
            f"anchor={record['anchor_score']:.3f} "
            f"{'PASS' if record['top2_passed'] else 'FAIL'}"
        )

    rows = [completed[position] for position in sorted(completed)]
    if len(rows) != 49:
        raise RuntimeError(f"Only {len(rows)}/49 short rollouts completed.")

    # Fixed, label-free combination: equal mean of three within-task ranks.
    # Outcomes are not accessed until the ranking has been computed.
    component_fields = [
        "persistence_after_forced",
        "weighted_semantic_divergence",
        "anchor_score",
    ]
    component_scores = {
        field: ordinal_percentile_scores(rows, field, descending=True)
        for field in component_fields
    }
    for row in rows:
        position = int(row["position"])
        row["semantic_lookahead_score"] = float(
            np.mean([
                component_scores[field][position]
                for field in component_fields
            ])
        )

    selectors = {
        "entropy_high": ("same_run_entropy", True),
        "probability_margin_low": ("same_run_probability_margin", False),
        "edit_distance_high": ("normalized_edit_distance", True),
        "mismatch_high": ("aligned_mismatch", True),
        "persistence_high": ("persistence_after_forced", True),
        "weighted_semantic_high": (
            "weighted_semantic_divergence",
            True,
        ),
        "anchor_score_high": ("anchor_score", True),
        "semantic_lookahead_high": ("semantic_lookahead_score", True),
    }
    metrics = {
        name: ranking_metrics(rows, field, descending)
        for name, (field, descending) in selectors.items()
    }
    same_run_pair_mismatches = [
        int(row["position"])
        for row in rows
        if not row["saved_pair_matches_same_run"]
    ]
    rollout_mismatches = [
        int(row["position"])
        for row in rows
        if not row["short_rollout_matches_saved_branch"]
    ]

    report = {
        "status": "descriptive_single_problem_pilot",
        "task_id": LOOKAHEAD_TASK_ID,
        "model": LOOKAHEAD_MODEL,
        "evaluated_positions": len(rows),
        "recoveries": sum(int(row["top2_passed"]) for row in rows),
        "lookahead_tokens": LOOKAHEAD_TOKENS,
        "selection_uses_execution_tests": False,
        "combination_rule": {
            "type": "equal_mean_of_within_problem_ordinal_percentiles",
            "components": component_fields,
            "weights": [1 / 3, 1 / 3, 1 / 3],
            "weights_fitted_on_labels": False,
        },
        "metrics": metrics,
        "saved_pair_mismatch_positions": same_run_pair_mismatches,
        "saved_branch_rollout_mismatch_positions": rollout_mismatches,
        "decision_rule_for_next_stage": {
            "continue_if": (
                "semantic_lookahead finds >=1 recovery in top-5 AND "
                "beats entropy top-5; otherwise redesign the signal"
            ),
            "warning": (
                "This rule only checks plausibility on one problem and is "
                "not evidence of generalization."
            ),
        },
    }
    write_csv(rows, OUTPUT_DIR / "semantic_lookahead_positions.csv")
    (OUTPUT_DIR / "semantic_lookahead_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    make_plot(rows, OUTPUT_DIR / "semantic_lookahead_rankings.png")

    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    archive_base = str(OUTPUT_ZIP.with_suffix(""))
    shutil.make_archive(archive_base, "zip", root_dir=OUTPUT_DIR)

    print("\n=== SEMANTIC LOOKAHEAD REPORT ===")
    compact = {
        name: {
            "AUROC": round(values["auroc"], 3),
            "AP": round(values["average_precision"], 3),
            "positive_ranks": values["positive_ranks"],
            "top5_recoveries": values["top_k"]["5"]["recoveries"],
        }
        for name, values in metrics.items()
    }
    print(json.dumps(compact, indent=2, ensure_ascii=False))
    print("\nDownload this ZIP and send it back:")
    print(OUTPUT_ZIP)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


run_semantic_lookahead_pilot(overwrite=False)

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[ 1/49] t= 2 cached
[ 2/49] t= 3 cached
[ 3/49] t= 4 cached
[ 4/49] t= 5 cached
[ 5/49] t= 6 cached
[ 6/49] t= 7 cached
[ 7/49] t= 8 cached
[ 8/49] t= 9 cached
[ 9/49] t=10 cached
[10/49] t=11 cached
[11/49] t=12 cached
[12/49] t=13 cached
[13/49] t=14 cached
[14/49] t=15 cached
[15/49] t=16 cached
[16/49] t=17 cached
[17/49] t=18 cached
[18/49] t=19 cached
[19/49] t=20 cached
[20/49] t=21 cached
[21/49] t=22 cached
[22/49] t=23 cached
[23/49] t=24 cached
[24/49] t=25 cached
[25/49] t=26 cached
[26/49] t=27 cached
[27/49] t=28 cached
[28/49] t=29 cached
[29/49] t=30 cached
[30/49] t=31 cached
[31/49] t=32 cached
[32/49] t=33 cached
[33/49] t=34 cached
[34/49] t=35 cached
[35/49] t=36 cached
[36/49] t=37 cached
[37/49] t=38 cached
[38/49] t=39 cached
[39/49] t=40 cached
[40/49] t=41 cached
[41/49] t=42 cached
[42/49] t=43 cached
[43/49] t=44 cached
[44/49] t=45 cached
[45/49] t=46 cached
[46/49] t=47 cached
[47/49] t=48 cached
[48/49] t=49 cached
[49/49] t=50 cached

=== SEMANTIC LOOKAH

In [5]:
"""Frozen multi-problem validation of semantic top-2 branching.

Kaggle usage
------------
Run the dependency cell and the large definitions cell from
``notebook0cef4050e3.ipynb``, then paste this complete file into a new cell.

The experiment deliberately excludes HumanEval/26, which was used to design
the selector. It deterministically scans HumanEval tasks until it obtains 20
greedy-baseline failures. For each failure it:

1. evaluates every position with a full 12-token baseline horizon;
2. computes the frozen semantic-lookahead score using tie-aware midranks;
3. selects top-5 positions using semantic lookahead, entropy, probability
   margin, and a deterministic random control;
4. fully generates and evaluates the union of those positions.

Tests are never used to select a position. Outputs are incremental and can be
resumed. The final archive also contains EvalPlus-compatible JSONL files.
"""

from __future__ import annotations

import ast
import csv
import gc
import hashlib
import json
import keyword
import math
import re
import shutil
import time
import zipfile
from collections import defaultdict
from dataclasses import asdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# ---------------------------------------------------------------------------
# Frozen protocol. Changing any value creates a different experiment.
# ---------------------------------------------------------------------------

VALIDATION_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEVELOPMENT_TASK_ID = "HumanEval/26"
TARGET_BASELINE_FAILURES = 20
BRANCH_BUDGET = 5
LOOKAHEAD_TOKENS = 12
EDGE_BUFFER = 2
MAX_NEW_TOKENS = 256
TEST_TIMEOUT_S = 12
COHORT_SEED = 20260730
RANDOM_CONTROL_SEED = 42026

OUTPUT_DIR = Path("/kaggle/working/frozen_selector_validation")
OUTPUT_ZIP = Path("/kaggle/working/frozen_selector_validation_checkpoint.zip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINES_PATH = OUTPUT_DIR / "baselines.jsonl"
LOOKAHEADS_PATH = OUTPUT_DIR / "lookaheads.jsonl"
SELECTIONS_PATH = OUTPUT_DIR / "selections.jsonl"
BRANCHES_PATH = OUTPUT_DIR / "branches.jsonl"
METADATA_PATH = OUTPUT_DIR / "metadata.json"

SELECTORS = (
    "semantic_lookahead",
    "entropy",
    "probability_margin",
    "random",
)
SEMANTIC_COMPONENTS = (
    "persistence_after_forced",
    "weighted_semantic_divergence",
    "anchor_score",
)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()


def stable_task_number(task_id: str) -> int:
    digest = hashlib.sha256(task_id.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big")


def restore_checkpoint_if_available() -> None:
    """Restore an uploaded prior checkpoint only when local output is empty."""
    if any(path.exists() for path in (
        BASELINES_PATH,
        LOOKAHEADS_PATH,
        SELECTIONS_PATH,
        BRANCHES_PATH,
    )):
        return

    candidates = []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(
            input_root.rglob("frozen_selector_validation_checkpoint.zip")
        )
    if not candidates:
        return

    archive_path = sorted(candidates, key=lambda path: len(str(path)))[0]
    print(f"Restoring checkpoint: {archive_path}")
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            path = Path(info.filename)
            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError(f"Unsafe checkpoint path: {info.filename}")
        archive.extractall(OUTPUT_DIR)


def save_checkpoint() -> str:
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    return shutil.make_archive(
        str(OUTPUT_ZIP.with_suffix("")),
        "zip",
        root_dir=OUTPUT_DIR,
    )


def frozen_config() -> dict[str, Any]:
    return {
        "protocol_version": 1,
        "model": VALIDATION_MODEL,
        "development_task_excluded": DEVELOPMENT_TASK_ID,
        "target_baseline_failures": TARGET_BASELINE_FAILURES,
        "branch_budget": BRANCH_BUDGET,
        "lookahead_tokens": LOOKAHEAD_TOKENS,
        "edge_buffer": EDGE_BUFFER,
        "max_new_tokens": MAX_NEW_TOKENS,
        "test_timeout_s": TEST_TIMEOUT_S,
        "cohort_seed": COHORT_SEED,
        "random_control_seed": RANDOM_CONTROL_SEED,
        "selectors": list(SELECTORS),
        "semantic_components": list(SEMANTIC_COMPONENTS),
        "semantic_weights": [1 / 3, 1 / 3, 1 / 3],
        "tie_break_rule": "seeded_sha256_of_task_selector_position",
        "selection_uses_tests": False,
    }


def validate_or_write_metadata(task_order: list[str]) -> None:
    expected = {
        "config": frozen_config(),
        "candidate_task_order": task_order,
    }
    if METADATA_PATH.exists():
        existing = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
        if existing != expected:
            raise RuntimeError(
                "Checkpoint configuration differs from this frozen protocol. "
                "Use a new output directory instead of mixing experiments."
            )
    else:
        METADATA_PATH.write_text(
            json.dumps(expected, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )


# ---------------------------------------------------------------------------
# Label-free short-trajectory features.
# ---------------------------------------------------------------------------

def token_class(text: str) -> str:
    if not text:
        return "empty"
    if text.isspace() or "\n" in text:
        return "whitespace"
    stripped = text.strip()
    if keyword.iskeyword(stripped):
        return "keyword"
    if stripped.isidentifier():
        return "identifier"
    if stripped in {
        "+", "-", "*", "/", "//", "%", "**", "=", "==", "!=", "<", ">",
        "<=", ">=", ":=", "&", "|", "^", "~", "<<", ">>", "->",
    }:
        return "operator"
    try:
        ast.literal_eval(stripped)
        return "literal"
    except Exception:
        return "other"


def levenshtein_distance(left: list[int], right: list[int]) -> int:
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right) + 1))
    for row_index, left_value in enumerate(left, start=1):
        current = [row_index]
        for column_index, right_value in enumerate(right, start=1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1]
                    + (left_value != right_value),
                )
            )
        previous = current
    return previous[-1]


def aligned_mismatch(
    left: list[int],
    right: list[int],
    start: int = 0,
) -> float:
    length = max(len(left), len(right))
    if length <= start:
        return 0.0
    mismatches = 0
    for index in range(start, length):
        left_value = left[index] if index < len(left) else None
        right_value = right[index] if index < len(right) else None
        mismatches += left_value != right_value
    return mismatches / (length - start)


def weighted_token_divergence(
    baseline_texts: list[str],
    branch_texts: list[str],
) -> float:
    length = max(len(baseline_texts), len(branch_texts))
    if length == 0:
        return 0.0
    total = 0.0
    for index in range(length):
        left = baseline_texts[index] if index < len(baseline_texts) else ""
        right = branch_texts[index] if index < len(branch_texts) else ""
        if left == right:
            continue
        left_class = token_class(left)
        right_class = token_class(right)
        classes = {left_class, right_class}
        if classes == {"whitespace"}:
            weight = 0.0
        elif classes == {"identifier"}:
            weight = 0.20
        elif "whitespace" in classes:
            weight = 0.25
        elif classes & {"keyword", "operator", "literal"}:
            weight = 1.0
        else:
            weight = 0.50
        total += weight
    return total / length


ANCHOR_OPERATOR_PATTERN = re.compile(
    r"==|!=|<=|>=|:=|//|\*\*|->|[+\-*/%<>=&|^~]"
)
ANCHOR_CALL_PATTERN = re.compile(r"\b([A-Za-z_]\w*)\s*\(")
ANCHOR_METHOD_PATTERN = re.compile(r"\.\s*([A-Za-z_]\w*)\b")
ANCHOR_NUMBER_PATTERN = re.compile(
    r"(?<![\w.])(?:0[xob][0-9A-Fa-f]+|\d+(?:\.\d+)?)(?![\w.])"
)
ANCHOR_WORD_PATTERN = re.compile(r"\b[A-Za-z_]\w*\b")


def code_anchors(text: str) -> list[str]:
    anchors = []
    for word in ANCHOR_WORD_PATTERN.findall(text):
        if keyword.iskeyword(word):
            anchors.append(f"keyword:{word}")
    anchors.extend(
        f"operator:{value}" for value in ANCHOR_OPERATOR_PATTERN.findall(text)
    )
    anchors.extend(
        f"call:{value}" for value in ANCHOR_CALL_PATTERN.findall(text)
        if not keyword.iskeyword(value)
    )
    anchors.extend(
        f"method:{value}" for value in ANCHOR_METHOD_PATTERN.findall(text)
    )
    anchors.extend(
        f"number:{value}" for value in ANCHOR_NUMBER_PATTERN.findall(text)
    )
    return anchors


def anchor_features(
    baseline_text: str,
    branch_text: str,
) -> dict[str, float]:
    baseline_anchors = set(code_anchors(baseline_text))
    branch_anchors = set(code_anchors(branch_text))
    union = baseline_anchors | branch_anchors
    distance = (
        1.0
        - len(baseline_anchors & branch_anchors) / len(union)
        if union
        else 0.0
    )

    baseline_lexemes = max(
        len(ANCHOR_WORD_PATTERN.findall(baseline_text)),
        1,
    )
    branch_lexemes = max(
        len(ANCHOR_WORD_PATTERN.findall(branch_text)),
        1,
    )
    baseline_density = len(code_anchors(baseline_text)) / baseline_lexemes
    branch_density = len(code_anchors(branch_text)) / branch_lexemes
    density_retention = min(
        (branch_density + 0.05) / (baseline_density + 0.05),
        1.0,
    )
    return {
        "anchor_distance": distance,
        "baseline_anchor_density": baseline_density,
        "branch_anchor_density": branch_density,
        "anchor_score": distance * density_retention,
    }


def midrank_percentile_scores(
    rows: list[dict[str, Any]],
    field: str,
    descending: bool = True,
) -> dict[int, float]:
    """Tie-aware percentiles: equal values receive the same mean rank."""
    groups: dict[float, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        groups[float(row[field])].append(row)
    ordered_values = sorted(groups, reverse=descending)
    denominator = max(len(rows) - 1, 1)
    cursor = 1
    scores = {}
    for value in ordered_values:
        group = groups[value]
        mean_rank = cursor + (len(group) - 1) / 2
        score = 1.0 - (mean_rank - 1) / denominator
        for row in group:
            scores[int(row["position"])] = score
        cursor += len(group)
    return scores


# ---------------------------------------------------------------------------
# Model passes.
# ---------------------------------------------------------------------------

@torch.inference_mode()
def generate_short_forced_rollout(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
) -> dict[str, Any]:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    if prefix_ids:
        prefix_tensor = torch.tensor(
            [prefix_ids],
            dtype=torch.long,
            device=input_device,
        )
        context_ids = torch.cat([prompt_ids, prefix_tensor], dim=1)
    else:
        context_ids = prompt_ids

    started = time.perf_counter()
    outputs = model(context_ids, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[0, -1, :].float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    top_probabilities, top_ids = torch.topk(probabilities, k=2)
    entropy = float(
        -(probabilities * log_probabilities).sum().item()
    )

    branch_ids = [int(forced_token_id)]
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None or forced_token_id != eos_token_id:
        next_input = torch.tensor(
            [[forced_token_id]],
            dtype=torch.long,
            device=input_device,
        )
        outputs = model(next_input, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[0, -1, :]

        for _ in range(LOOKAHEAD_TOKENS - 1):
            next_id = int(torch.argmax(next_logits).item())
            branch_ids.append(next_id)
            if eos_token_id is not None and next_id == eos_token_id:
                break
            next_input = torch.tensor(
                [[next_id]],
                dtype=torch.long,
                device=input_device,
            )
            outputs = model(
                next_input,
                past_key_values=cache,
                use_cache=True,
            )
            cache = outputs.past_key_values
            next_logits = outputs.logits[0, -1, :]

    result = {
        "branch_token_ids": branch_ids,
        "branch_token_texts": [
            tokenizer.decode([token_id], skip_special_tokens=False)
            for token_id in branch_ids
        ],
        "same_run_top1_id": int(top_ids[0].item()),
        "same_run_top2_id": int(top_ids[1].item()),
        "same_run_p1": float(top_probabilities[0].item()),
        "same_run_p2": float(top_probabilities[1].item()),
        "probability_margin": float(
            top_probabilities[0].item() - top_probabilities[1].item()
        ),
        "entropy": entropy,
        "forced_probability": float(probabilities[forced_token_id].item()),
        "forced_rank": (
            int((logits > logits[forced_token_id]).sum().item()) + 1
        ),
        "lookahead_latency_s": time.perf_counter() - started,
    }
    del outputs, logits, log_probabilities, probabilities
    return result


@torch.inference_mode()
def generate_full_forced_branch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
) -> dict[str, Any]:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    generated_ids = list(prefix_ids) + [int(forced_token_id)]
    generated_tensor = torch.tensor(
        [generated_ids],
        dtype=torch.long,
        device=input_device,
    )
    input_ids = torch.cat([prompt_ids, generated_tensor], dim=1)
    eos_token_id = tokenizer.eos_token_id
    started = time.perf_counter()

    if eos_token_id is None or forced_token_id != eos_token_id:
        outputs = model(input_ids, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[0, -1, :]
        remaining = max(MAX_NEW_TOKENS - len(generated_ids), 0)
        for _ in range(remaining):
            next_id = int(torch.argmax(next_logits).item())
            generated_ids.append(next_id)
            if eos_token_id is not None and next_id == eos_token_id:
                break
            next_input = torch.tensor(
                [[next_id]],
                dtype=torch.long,
                device=input_device,
            )
            outputs = model(
                next_input,
                past_key_values=cache,
                use_cache=True,
            )
            cache = outputs.past_key_values
            next_logits = outputs.logits[0, -1, :]

    return {
        "token_ids": generated_ids,
        "code": tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ),
        "generation_latency_s": time.perf_counter() - started,
    }


# ---------------------------------------------------------------------------
# Cohort, ranking, evaluation, and reporting.
# ---------------------------------------------------------------------------

def ordered_validation_tasks(all_tasks: list[Any]) -> list[Any]:
    candidates = [
        task for task in all_tasks
        if task.task_id != DEVELOPMENT_TASK_ID
    ]
    rng = np.random.default_rng(COHORT_SEED)
    order = rng.permutation(len(candidates))
    return [candidates[int(index)] for index in order]


def baseline_to_record(
    task: Any,
    generation: Any,
    passed: bool,
    error: str | None,
) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": bool(passed),
        "error": error,
        "code": generation.code,
        "token_ids": [int(value) for value in generation.token_ids],
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def eligible_positions(baseline: dict[str, Any]) -> list[int]:
    """Require the same full 12-token baseline horizon at every position."""
    token_count = len(baseline["token_ids"])
    final_start = token_count - LOOKAHEAD_TOKENS
    if final_start < EDGE_BUFFER:
        return []
    return list(range(EDGE_BUFFER, final_start + 1))


def make_lookahead_record(
    task_id: str,
    position: int,
    baseline: dict[str, Any],
    generated: dict[str, Any],
    tokenizer: Any,
) -> dict[str, Any]:
    trace = baseline["trace"][position]
    baseline_ids = baseline["token_ids"][
        position:position + LOOKAHEAD_TOKENS
    ]
    branch_ids = generated["branch_token_ids"]
    baseline_texts = [
        tokenizer.decode([token_id], skip_special_tokens=False)
        for token_id in baseline_ids
    ]
    baseline_snippet = tokenizer.decode(
        baseline_ids,
        skip_special_tokens=True,
    )
    branch_snippet = tokenizer.decode(
        branch_ids,
        skip_special_tokens=True,
    )
    saved_pair = {int(trace["top1_id"]), int(trace["top2_id"])}
    same_run_pair = {
        generated["same_run_top1_id"],
        generated["same_run_top2_id"],
    }
    return {
        "task_id": task_id,
        "position": position,
        "top1_token_id": int(trace["top1_id"]),
        "top2_token_id": int(trace["top2_id"]),
        "top1_token": trace["top1_text"],
        "top2_token": trace["top2_text"],
        "baseline_token_ids": baseline_ids,
        "baseline_snippet": baseline_snippet,
        "branch_snippet": branch_snippet,
        "normalized_edit_distance": levenshtein_distance(
            baseline_ids,
            branch_ids,
        ) / max(len(baseline_ids), len(branch_ids), 1),
        "aligned_mismatch": aligned_mismatch(
            baseline_ids,
            branch_ids,
        ),
        "persistence_after_forced": aligned_mismatch(
            baseline_ids,
            branch_ids,
            start=1,
        ),
        "weighted_semantic_divergence": weighted_token_divergence(
            baseline_texts,
            generated["branch_token_texts"],
        ),
        **anchor_features(baseline_snippet, branch_snippet),
        **generated,
        "saved_pair_matches_same_run": saved_pair == same_run_pair,
    }


def add_semantic_scores(
    rows: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    component_scores = {
        field: midrank_percentile_scores(rows, field, descending=True)
        for field in SEMANTIC_COMPONENTS
    }
    for row in rows:
        position = int(row["position"])
        row["semantic_lookahead_score"] = float(np.mean([
            component_scores[field][position]
            for field in SEMANTIC_COMPONENTS
        ]))
    return rows


def deterministic_random_positions(
    task_id: str,
    positions: list[int],
) -> list[int]:
    seed = (
        RANDOM_CONTROL_SEED
        + stable_task_number(task_id)
    ) % (2**63 - 1)
    rng = np.random.default_rng(seed)
    count = min(BRANCH_BUDGET, len(positions))
    return sorted(
        int(value)
        for value in rng.choice(positions, size=count, replace=False)
    )


def deterministic_tie_break(
    task_id: str,
    selector: str,
    position: int,
) -> int:
    payload = (
        f"{RANDOM_CONTROL_SEED}|{task_id}|{selector}|{position}"
    ).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big")


def select_task_positions(
    task_id: str,
    rows: list[dict[str, Any]],
) -> dict[str, Any]:
    if len(rows) < BRANCH_BUDGET:
        raise RuntimeError(
            f"{task_id} has only {len(rows)} eligible positions."
        )
    semantic = [
        int(row["position"])
        for row in sorted(
            rows,
            key=lambda row: (
                -float(row["semantic_lookahead_score"]),
                deterministic_tie_break(
                    task_id,
                    "semantic_lookahead",
                    int(row["position"]),
                ),
            ),
        )[:BRANCH_BUDGET]
    ]
    entropy = [
        int(row["position"])
        for row in sorted(
            rows,
            key=lambda row: (
                -float(row["entropy"]),
                deterministic_tie_break(
                    task_id,
                    "entropy",
                    int(row["position"]),
                ),
            ),
        )[:BRANCH_BUDGET]
    ]
    margin = [
        int(row["position"])
        for row in sorted(
            rows,
            key=lambda row: (
                float(row["probability_margin"]),
                deterministic_tie_break(
                    task_id,
                    "probability_margin",
                    int(row["position"]),
                ),
            ),
        )[:BRANCH_BUDGET]
    ]
    random_positions = deterministic_random_positions(
        task_id,
        [int(row["position"]) for row in rows],
    )
    by_selector = {
        "semantic_lookahead": semantic,
        "entropy": entropy,
        "probability_margin": margin,
        "random": random_positions,
    }
    union = sorted({
        position
        for positions in by_selector.values()
        for position in positions
    })
    return {
        "task_id": task_id,
        "budget_per_selector": BRANCH_BUDGET,
        "by_selector": by_selector,
        "union_positions": union,
        "eligible_positions": len(rows),
    }


def wilson_interval(successes: int, total: int) -> list[float]:
    if total == 0:
        return [float("nan"), float("nan")]
    z = 1.959963984540054
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (
        proportion + z * z / (2 * total)
    ) / denominator
    half_width = (
        z
        * math.sqrt(
            proportion * (1 - proportion) / total
            + z * z / (4 * total * total)
        )
        / denominator
    )
    return [max(0.0, center - half_width), min(1.0, center + half_width)]


def exact_mcnemar_pvalue(
    semantic_only: int,
    comparator_only: int,
) -> float:
    discordant = semantic_only + comparator_only
    if discordant == 0:
        return 1.0
    lower = min(semantic_only, comparator_only)
    one_tail = sum(
        math.comb(discordant, value)
        for value in range(lower + 1)
    ) / (2**discordant)
    return min(1.0, 2 * one_tail)


def export_evalplus_samples(
    complete_task_ids: list[str],
    tasks_by_id: dict[str, Any],
    selections_by_task: dict[str, dict[str, Any]],
    branches_by_key: dict[tuple[str, int], dict[str, Any]],
) -> None:
    for selector in SELECTORS:
        path = OUTPUT_DIR / f"evalplus_samples_{selector}.jsonl"
        with path.open("w", encoding="utf-8") as handle:
            for task_id in complete_task_ids:
                task = tasks_by_id[task_id]
                for position in selections_by_task[task_id][
                    "by_selector"
                ][selector]:
                    branch = branches_by_key[(task_id, position)]
                    sample = {
                        "task_id": task_id,
                        "solution": extract_code(
                            branch["code"],
                            task.entry_point,
                        ),
                    }
                    handle.write(
                        json.dumps(sample, ensure_ascii=False) + "\n"
                    )


def write_report(
    failure_task_ids: list[str],
    tasks_by_id: dict[str, Any],
) -> dict[str, Any]:
    selections_by_task = {
        row["task_id"]: row for row in read_jsonl(SELECTIONS_PATH)
    }
    branches_by_key = {
        (row["task_id"], int(row["position"])): row
        for row in read_jsonl(BRANCHES_PATH)
    }
    complete_task_ids = []
    for task_id in failure_task_ids:
        selection = selections_by_task.get(task_id)
        if selection is None:
            continue
        if all(
            (task_id, position) in branches_by_key
            for position in selection["union_positions"]
        ):
            complete_task_ids.append(task_id)

    selector_success: dict[str, dict[str, bool]] = {
        selector: {} for selector in SELECTORS
    }
    selector_rows = {}
    for selector in SELECTORS:
        total = len(complete_task_ids)
        successes = 0
        evaluated_branches = 0
        for task_id in complete_task_ids:
            positions = selections_by_task[task_id][
                "by_selector"
            ][selector]
            passed = any(
                branches_by_key[(task_id, position)]["passed"]
                for position in positions
            )
            selector_success[selector][task_id] = bool(passed)
            successes += int(passed)
            evaluated_branches += len(positions)
        selector_rows[selector] = {
            "complete_tasks": total,
            "recovered_tasks": successes,
            "recovery_rate": successes / total if total else None,
            "wilson_95_interval": (
                wilson_interval(successes, total) if total else None
            ),
            "nominal_evaluated_branches": evaluated_branches,
        }

    paired_comparisons = {}
    for comparator in SELECTORS:
        if comparator == "semantic_lookahead":
            continue
        semantic_only = sum(
            selector_success["semantic_lookahead"][task_id]
            and not selector_success[comparator][task_id]
            for task_id in complete_task_ids
        )
        comparator_only = sum(
            selector_success[comparator][task_id]
            and not selector_success["semantic_lookahead"][task_id]
            for task_id in complete_task_ids
        )
        paired_comparisons[f"semantic_vs_{comparator}"] = {
            "semantic_only": semantic_only,
            "comparator_only": comparator_only,
            "both": sum(
                selector_success["semantic_lookahead"][task_id]
                and selector_success[comparator][task_id]
                for task_id in complete_task_ids
            ),
            "neither": sum(
                not selector_success["semantic_lookahead"][task_id]
                and not selector_success[comparator][task_id]
                for task_id in complete_task_ids
            ),
            "exact_mcnemar_two_sided_p": exact_mcnemar_pvalue(
                semantic_only,
                comparator_only,
            ),
        }

    report = {
        "status": (
            "complete"
            if len(complete_task_ids) == TARGET_BASELINE_FAILURES
            else "partial"
        ),
        "config": frozen_config(),
        "baseline_failures_selected": failure_task_ids,
        "complete_task_ids": complete_task_ids,
        "complete_tasks": len(complete_task_ids),
        "selector_results_on_base_humaneval_tests": selector_rows,
        "paired_problem_level_comparisons": paired_comparisons,
        "unique_full_branches_evaluated": len(branches_by_key),
        "all_saved_pairs_reproduced": all(
            row["saved_pair_matches_same_run"]
            for row in read_jsonl(LOOKAHEADS_PATH)
        ),
        "interpretation_limits": [
            "HumanEval/26 was excluded because it was used for feature design.",
            "The in-notebook pass/fail result uses base HumanEval tests.",
            "Run the exported samples with EvalPlus before publication.",
            "Statistical tests are descriptive with only 20 problems.",
        ],
    }
    (OUTPUT_DIR / "validation_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    if complete_task_ids:
        export_evalplus_samples(
            complete_task_ids,
            tasks_by_id,
            selections_by_task,
            branches_by_key,
        )
    return report


def write_compact_csv() -> None:
    rows = read_jsonl(BRANCHES_PATH)
    fields = [
        "task_id",
        "position",
        "selected_by",
        "passed",
        "top1_token",
        "top2_token",
        "semantic_lookahead_score",
        "entropy",
        "probability_margin",
        "generation_latency_s",
        "error",
    ]
    with (OUTPUT_DIR / "branch_results.csv").open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in sorted(
            rows,
            key=lambda item: (item["task_id"], item["position"]),
        ):
            output = {field: row.get(field) for field in fields}
            output["selected_by"] = ",".join(row["selected_by"])
            writer.writerow(output)


def run_frozen_validation() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator in Kaggle.")

    restore_checkpoint_if_available()
    torch.manual_seed(COHORT_SEED)
    np.random.seed(COHORT_SEED)

    all_tasks = load_tasks("", num_tasks=164)
    tasks_by_id = {task.task_id: task for task in all_tasks}
    ordered_tasks = ordered_validation_tasks(all_tasks)
    task_order = [task.task_id for task in ordered_tasks]
    validate_or_write_metadata(task_order)

    gc.collect()
    torch.cuda.empty_cache()
    print(
        f"Loading {VALIDATION_MODEL} on "
        f"{torch.cuda.device_count()} GPU(s)..."
    )
    tokenizer = AutoTokenizer.from_pretrained(VALIDATION_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        VALIDATION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    # Stage 1: deterministic baseline-failure cohort.
    baselines_by_task = {
        row["task_id"]: row for row in read_jsonl(BASELINES_PATH)
    }
    failure_task_ids = []
    print("\n=== STAGE 1: BASELINE COHORT ===")
    for task in ordered_tasks:
        if len(failure_task_ids) >= TARGET_BASELINE_FAILURES:
            break
        if task.task_id not in baselines_by_task:
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(
                model,
                tokenizer,
                prompt,
                MAX_NEW_TOKENS,
            )
            passed, error = evaluate(
                task,
                generation.code,
                TEST_TIMEOUT_S,
            )
            record = baseline_to_record(
                task,
                generation,
                passed,
                error,
            )
            append_jsonl(BASELINES_PATH, record)
            baselines_by_task[task.task_id] = record
            print(
                f"{task.task_id}: {'PASS' if passed else 'FAIL'} "
                f"({len(generation.token_ids)} tokens)"
            )
        baseline = baselines_by_task[task.task_id]
        if not baseline["passed"]:
            if len(eligible_positions(baseline)) >= BRANCH_BUDGET:
                failure_task_ids.append(task.task_id)
            else:
                print(
                    f"{task.task_id}: failed but excluded because it has "
                    "fewer than five full-horizon positions."
                )

    if len(failure_task_ids) < TARGET_BASELINE_FAILURES:
        raise RuntimeError(
            f"Found only {len(failure_task_ids)} eligible baseline failures."
        )
    print(f"Frozen failure cohort: {failure_task_ids}")
    print(f"Baseline checkpoint: {save_checkpoint()}")

    # Stage 2 and 3: lookahead, fixed selection, full branches.
    lookaheads_by_key = {
        (row["task_id"], int(row["position"])): row
        for row in read_jsonl(LOOKAHEADS_PATH)
    }
    selections_by_task = {
        row["task_id"]: row for row in read_jsonl(SELECTIONS_PATH)
    }
    branches_by_key = {
        (row["task_id"], int(row["position"])): row
        for row in read_jsonl(BRANCHES_PATH)
    }

    for task_number, task_id in enumerate(failure_task_ids, start=1):
        task = tasks_by_id[task_id]
        baseline = baselines_by_task[task_id]
        prompt = build_prompt(task, tokenizer)
        positions = eligible_positions(baseline)
        print(
            f"\n=== TASK {task_number}/{TARGET_BASELINE_FAILURES}: "
            f"{task_id} ({len(positions)} positions) ==="
        )

        for position_index, position in enumerate(positions, start=1):
            key = (task_id, position)
            if key in lookaheads_by_key:
                continue
            trace = baseline["trace"][position]
            generated = generate_short_forced_rollout(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                prefix_ids=baseline["token_ids"][:position],
                forced_token_id=int(trace["top2_id"]),
            )
            record = make_lookahead_record(
                task_id,
                position,
                baseline,
                generated,
                tokenizer,
            )
            append_jsonl(LOOKAHEADS_PATH, record)
            lookaheads_by_key[key] = record
            if position_index % 10 == 0 or position_index == len(positions):
                print(
                    f"  lookahead {position_index}/{len(positions)}"
                )

        task_lookaheads = add_semantic_scores([
            dict(lookaheads_by_key[(task_id, position)])
            for position in positions
        ])
        if task_id not in selections_by_task:
            selection = select_task_positions(task_id, task_lookaheads)
            append_jsonl(SELECTIONS_PATH, selection)
            selections_by_task[task_id] = selection
        selection = selections_by_task[task_id]
        print(f"  selected: {selection['by_selector']}")
        rows_by_position = {
            int(row["position"]): row for row in task_lookaheads
        }

        for position in selection["union_positions"]:
            key = (task_id, int(position))
            if key in branches_by_key:
                continue
            lookahead = rows_by_position[int(position)]
            generated = generate_full_forced_branch(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                prefix_ids=baseline["token_ids"][:int(position)],
                forced_token_id=int(lookahead["top2_token_id"]),
            )
            passed, error = evaluate(
                task,
                generated["code"],
                TEST_TIMEOUT_S,
            )
            selected_by = [
                selector
                for selector, selector_positions
                in selection["by_selector"].items()
                if int(position) in selector_positions
            ]
            record = {
                "task_id": task_id,
                "position": int(position),
                "selected_by": selected_by,
                "passed": bool(passed),
                "error": error,
                "top1_token": lookahead["top1_token"],
                "top2_token": lookahead["top2_token"],
                "top1_token_id": lookahead["top1_token_id"],
                "top2_token_id": lookahead["top2_token_id"],
                "semantic_lookahead_score": lookahead[
                    "semantic_lookahead_score"
                ],
                "entropy": lookahead["entropy"],
                "probability_margin": lookahead["probability_margin"],
                **generated,
            }
            append_jsonl(BRANCHES_PATH, record)
            branches_by_key[key] = record
            print(
                f"  branch t={position:3d} "
                f"{'PASS' if passed else 'FAIL'} "
                f"[{','.join(selected_by)}]"
            )

        report = write_report(
            failure_task_ids,
            tasks_by_id,
        )
        write_compact_csv()
        checkpoint = save_checkpoint()
        print(
            f"  checkpoint: {report['complete_tasks']}/"
            f"{TARGET_BASELINE_FAILURES} tasks -> {checkpoint}"
        )

    report = write_report(failure_task_ids, tasks_by_id)
    write_compact_csv()
    checkpoint = save_checkpoint()

    print("\n=== FINAL REPORT ===")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print("\nDownload and send back:")
    print(checkpoint)
    print("\nFor strengthened HumanEval+ evaluation, run in a CPU cell:")
    print('!pip install -U "evalplus>=0.2.0"')
    for selector in SELECTORS:
        samples = OUTPUT_DIR / f"evalplus_samples_{selector}.jsonl"
        print(
            "!evalplus.evaluate --dataset humaneval "
            f'--samples "{samples}" --parallel 2'
        )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


run_frozen_validation()


Loading Qwen/Qwen2.5-Coder-7B-Instruct on 2 GPU(s)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


=== STAGE 1: BASELINE COHORT ===
HumanEval/120: failed but excluded because it has fewer than five full-horizon positions.
Frozen failure cohort: ['HumanEval/140', 'HumanEval/93', 'HumanEval/10', 'HumanEval/77', 'HumanEval/141', 'HumanEval/154', 'HumanEval/50', 'HumanEval/145', 'HumanEval/163', 'HumanEval/81', 'HumanEval/130', 'HumanEval/121', 'HumanEval/38', 'HumanEval/127', 'HumanEval/142', 'HumanEval/32', 'HumanEval/137', 'HumanEval/33', 'HumanEval/65', 'HumanEval/95']
Baseline checkpoint: /kaggle/working/frozen_selector_validation_checkpoint.zip

=== TASK 1/20: HumanEval/140 (8 positions) ===
  selected: {'semantic_lookahead': [6, 8, 7, 4, 3], 'entropy': [7, 6, 4, 5, 8], 'probability_margin': [7, 6, 4, 5, 8], 'random': [2, 6, 7, 8, 9]}
  checkpoint: 20/20 tasks -> /kaggle/working/frozen_selector_validation_checkpoint.zip

=== TASK 2/20: HumanEval/93 (80 positions) ===
  selected: {'semantic_lookahead': [62, 50, 58, 64, 65], 'entropy': [12, 66, 15, 5, 67], 'probability_margin': [12

In [6]:
# Agregar después de la celda run_frozen_validation().
# Ensayo exploratorio: 50% ranking de entropía + 50% ranking semántico.
import json, hashlib, math, gc, time
from pathlib import Path
from collections import defaultdict


def probar_hibrido():
    required = ['frozen_config', 'read_jsonl', 'append_jsonl',
                'add_semantic_scores', 'midrank_percentile_scores',
                'deterministic_tie_break', 'generate_full_forced_branch',
                'load_tasks', 'build_prompt', 'evaluate', 'extract_code']
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Ejecutá las definiciones de la validación: ' + ', '.join(missing))

    source = Path(OUTPUT_DIR)
    paths = {name: source / name for name in [
        'metadata.json', 'validation_report.json', 'baselines.jsonl',
        'lookaheads.jsonl', 'selections.jsonl', 'branches.jsonl']}
    for path in paths.values():
        if not path.exists():
            raise FileNotFoundError(f'Falta el checkpoint original: {path}')
    metadata = json.loads(paths['metadata.json'].read_text())
    report = json.loads(paths['validation_report.json'].read_text())
    if metadata['config'] != frozen_config():
        raise RuntimeError('La configuración en memoria difiere del checkpoint.')
    if report['status'] != 'complete':
        raise RuntimeError('Primero completá la validación original.')
    task_ids = report['complete_task_ids']
    k = int(metadata['config']['branch_budget'])
    if len(task_ids) != TARGET_BASELINE_FAILURES or len(set(task_ids)) != len(task_ids):
        raise RuntimeError('La cohorte no coincide con la validación original.')

    baseline = {r['task_id']: r for r in read_jsonl(paths['baselines.jsonl'])}
    original = {r['task_id']: r for r in read_jsonl(paths['selections.jsonl'])}
    lookaheads = defaultdict(list)
    for row in read_jsonl(paths['lookaheads.jsonl']):
        lookaheads[row['task_id']].append(row)

    # Congelar fuentes y pesos para reanudar sin mezclar experimentos.
    protocol = {'version': 1, 'entropy_weight': 0.5, 'semantic_weight': 0.5,
                'budget': k, 'task_ids': task_ids, 'exploratory': True,
                'source_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest()
                                  for name, path in paths.items()}}
    fingerprint = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:12]
    out = source.parent / f'hybrid_entropy_semantic_{fingerprint}'
    out.mkdir(exist_ok=True)
    (out / 'protocol.json').write_text(json.dumps(protocol, indent=2))
    new_branches_path = out / 'new_branches.jsonl'

    selected, by_position = {}, {}
    for tid in task_ids:
        if baseline[tid]['passed']:
            raise RuntimeError(f'{tid}: el baseline debería fallar.')
        rows = add_semantic_scores([dict(r) for r in lookaheads[tid]])
        positions = [int(r['position']) for r in rows]
        if len(set(positions)) != len(positions) or len(rows) != original[tid]['eligible_positions']:
            raise RuntimeError(f'{tid}: lookaheads incompletos o duplicados.')
        for row in rows:
            if not all(math.isfinite(float(row[f])) for f in ['entropy', 'semantic_lookahead_score']):
                raise RuntimeError(f'{tid}: puntaje no finito.')
        entropy_rank = midrank_percentile_scores(rows, 'entropy', descending=True)
        semantic_rank = midrank_percentile_scores(rows, 'semantic_lookahead_score', descending=True)
        for row in rows:
            pos = int(row['position'])
            row['hybrid_score'] = 0.5 * entropy_rank[pos] + 0.5 * semantic_rank[pos]
            by_position[(tid, pos)] = row
        ordered = sorted(rows, key=lambda r: (
            -r['hybrid_score'], deterministic_tie_break(tid, 'hybrid_50_50', int(r['position']))))
        selected[tid] = {s: list(v) for s, v in original[tid]['by_selector'].items()}
        selected[tid]['hybrid_50_50'] = [int(r['position']) for r in ordered[:k]]
        if any(len(v) != k or len(set(v)) != k for v in selected[tid].values()):
            raise RuntimeError(f'{tid}: presupuesto inconsistente.')
    (out / 'selections.json').write_text(json.dumps(selected, indent=2))

    # Leer resultados después de fijar la selección; nunca usarlos para ordenar.
    branches = {(r['task_id'], int(r['position'])): r
                for r in read_jsonl(paths['branches.jsonl'])}
    for tid in task_ids:
        for positions in original[tid]['by_selector'].values():
            if any((tid, int(pos)) not in branches for pos in positions):
                raise RuntimeError(f'{tid}: faltan ramas del ensayo original.')
    branches.update({(r['task_id'], int(r['position'])): r
                     for r in read_jsonl(new_branches_path)})
    needed = [(tid, pos) for tid in task_ids for pos in selected[tid]['hybrid_50_50']]
    for key in needed:
        if key in branches and int(branches[key]['top2_token_id']) != int(by_position[key]['top2_token_id']):
            raise RuntimeError(f'{key}: la rama guardada usa otro token.')
    pending = [key for key in needed if key not in branches]
    print(f'{len(task_ids)} problemas; k={k}. Ramas reutilizables: {len(needed)-len(pending)}; nuevas: {len(pending)}')

    if pending:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        if not torch.cuda.is_available():
            raise RuntimeError('Activá GPU para completar las ramas faltantes. La selección ya quedó guardada.')
        tasks = {t.task_id: t for t in load_tasks(','.join(task_ids), num_tasks=len(task_ids))}
        torch.manual_seed(COHORT_SEED)
        gc.collect()
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained(VALIDATION_MODEL)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            VALIDATION_MODEL, torch_dtype=torch.float16,
            device_map='auto', low_cpu_mem_usage=True)
        model.eval()
        try:
            for i, (tid, pos) in enumerate(pending, 1):
                row = by_position[(tid, pos)]
                generated = generate_full_forced_branch(
                    model=model, tokenizer=tokenizer,
                    prompt=build_prompt(tasks[tid], tokenizer),
                    prefix_ids=baseline[tid]['token_ids'][:pos],
                    forced_token_id=int(row['top2_token_id']))
                started = time.perf_counter()
                passed, error = evaluate(tasks[tid], generated['code'], TEST_TIMEOUT_S)
                record = {'task_id': tid, 'position': pos, 'passed': bool(passed),
                          'error': error, 'top2_token_id': int(row['top2_token_id']),
                          'evaluation_latency_s': time.perf_counter()-started, **generated}
                append_jsonl(new_branches_path, record)
                branches[(tid, pos)] = record
                print(f'{i}/{len(pending)} {tid} t={pos}: {"PASS" if passed else "FAIL"}', flush=True)
        finally:
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()

    selectors = list(selected[task_ids[0]])
    recovered = {s: {tid for tid in task_ids if any(
        branches[(tid, pos)]['passed'] for pos in selected[tid][s])} for s in selectors}
    summary = {s: {'recovered': len(recovered[s]), 'total': len(task_ids),
                   'rate': len(recovered[s])/len(task_ids)} for s in selectors}
    comparisons = {s: {'hybrid_only': sorted(recovered['hybrid_50_50']-recovered[s]),
                       'comparator_only': sorted(recovered[s]-recovered['hybrid_50_50'])}
                   for s in selectors if s != 'hybrid_50_50'}
    saved_new = read_jsonl(new_branches_path)
    result = {'exploratory': True, 'tests': 'HumanEval base', 'summary': summary,
              'paired_comparisons': comparisons,
              'semantic_only_vs_entropy': sorted(recovered['semantic_lookahead']-recovered['entropy']),
              'entropy_only_vs_semantic': sorted(recovered['entropy']-recovered['semantic_lookahead']),
              'incremental_new_branches': len(saved_new),
              'incremental_generation_seconds': sum(r['generation_latency_s'] for r in saved_new),
              'cost_note': 'Costo incremental del ensayo, no costo total del selector. Reutiliza lookaheads y ramas anteriores.'}
    (out / 'hybrid_report.json').write_text(json.dumps(result, indent=2, ensure_ascii=False))
    print('\nSelector                 Recuperados')
    for s, row in summary.items():
        print(f"{s:25} {row['recovered']}/{row['total']} ({row['rate']:.0%})")
    print('\nComparación por problema:', json.dumps(comparisons, indent=2))
    print('\nExploratorio: mismos problemas, tests base; pendiente validar en otros problemas y medir costo total.')
    print('Resultados:', out)
    return result

hybrid_result = probar_hibrido()


20 problemas; k=5. Ramas reutilizables: 79; nuevas: 21


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

1/21 HumanEval/93 t=61: FAIL
2/21 HumanEval/10 t=112: FAIL
3/21 HumanEval/10 t=145: FAIL
4/21 HumanEval/154 t=16: FAIL
5/21 HumanEval/50 t=4: FAIL
6/21 HumanEval/163 t=15: FAIL
7/21 HumanEval/81 t=35: PASS
8/21 HumanEval/81 t=34: PASS
9/21 HumanEval/130 t=103: FAIL
10/21 HumanEval/130 t=106: FAIL
11/21 HumanEval/130 t=64: FAIL
12/21 HumanEval/38 t=34: FAIL
13/21 HumanEval/38 t=87: FAIL
14/21 HumanEval/127 t=61: FAIL
15/21 HumanEval/32 t=27: FAIL
16/21 HumanEval/32 t=167: FAIL
17/21 HumanEval/137 t=15: PASS
18/21 HumanEval/137 t=11: PASS
19/21 HumanEval/33 t=149: PASS
20/21 HumanEval/65 t=26: FAIL
21/21 HumanEval/95 t=34: FAIL

Selector                 Recuperados
semantic_lookahead        10/20 (50%)
entropy                   10/20 (50%)
probability_margin        10/20 (50%)
random                    6/20 (30%)
hybrid_50_50              10/20 (50%)

Comparación por problema: {
  "semantic_lookahead": {
    "hybrid_only": [
      "HumanEval/95"
    ],
    "comparator_only": [
      "Hum

# Tito

In [ ]:
%pip uninstall -y torchvision
%pip install -q --upgrade --force-reinstall \
    "transformers==4.46.3" \
    "huggingface_hub==0.26.2" \
    "tokenizers==0.20.3" \
    "accelerate==0.34.2" \
    "bitsandbytes>=0.43.3"

import os
os.kill(os.getpid(), 9)

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires torchvision>=0.5, which is not installed.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.2 which is incompatible.
google-colab 1.0.0 requires jupy

In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_NEW_TOKENS = 20
TOP_K = 20

system_prompt = """Eres un evaluador académico experto en corrección de exámenes universitarios.

Recibirás una PREGUNTA, una RESPUESTA CORRECTA esperada y una RESPUESTA de un estudiante.
Tu tarea consiste en asignar la calificación final numérica exacta de 0 a 10.

Devuelve únicamente un número entero del 0 al 10.
No agregues explicaciones, comentarios ni markdown."""

user_prompt = """Pregunta: Describe cómo se pueden utilizar las aserciones en los tests y por qué son importantes.

Respuesta alumno: Las aserciones en los test se utilizan para ejecutar los test ya implementados. Se añaden, y se ejecuta para validar su funcionamiento. Es decir, por ejemplo, se añade un test, se implementa y al usar aserciones, se ejecuta, por lo tanto, se comprueba su comportamiento respecto al test.

Respuesta ideal: Las aserciones son declaraciones que verifican si una condición es verdadera en los tests. Son cruciales porque permiten a los desarrolladores comprobar el comportamiento del código bajo prueba, asegurando que produce los resultados esperados. En TDD, las aserciones guían el desarrollo al proporcionar retroalimentación inmediata sobre si el código cumple con los requisitos especificados. Utilizar una variedad de aserciones, como igualdad, nulidad y excepciones, enriquece el proceso de pruebas y ayuda a detectar errores de manera más efectiva."""

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
)
model.eval()

prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

device = model.get_input_embeddings().weight.device
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

rows, generated_ids = [], []

with torch.inference_mode():
    output = model(input_ids, use_cache=True)
    cache = output.past_key_values
    logits = output.logits[:, -1, :]

    for position in range(MAX_NEW_TOKENS):
        log_probs = torch.log_softmax(logits[0].float(), dim=-1)
        probabilities = log_probs.exp()
        top_probs, top_ids = torch.topk(probabilities, TOP_K)

        selected_id = int(top_ids[0].item())
        generated_ids.append(selected_id)

        for rank, (token_id, probability) in enumerate(
            zip(top_ids.tolist(), top_probs.tolist()), start=1
        ):
            rows.append({
                "position": position,
                "rank": rank,
                "token_id": token_id,
                "token": repr(tokenizer.decode([token_id], skip_special_tokens=False)),
                "probability": probability,
                "log_probability": float(log_probs[token_id].item()),
                "selected": token_id == selected_id,
            })

        if selected_id == tokenizer.eos_token_id:
            break

        output = model(
            torch.tensor([[selected_id]], dtype=torch.long, device=device),
            past_key_values=cache,
            use_cache=True,
        )
        cache = output.past_key_values
        logits = output.logits[:, -1, :]

answer = tokenizer.decode(generated_ids, skip_special_tokens=True)
distribution = pd.DataFrame(rows)

print("Respuesta generada:", repr(answer))
display(distribution)

distribution.to_csv("/kaggle/working/grading_distribution.csv", index=False)

with open("/kaggle/working/grading_distribution.json", "w", encoding="utf-8") as file:
    json.dump(
        {"answer": answer, "token_ids": generated_ids, "distribution": rows},
        file,
        ensure_ascii=False,
        indent=2,
    )

In [2]:
# ============================================================
# REVISIÓN: SEMANTIC LOOKAHEAD CON ALINEACIÓN DE TOKENS
# ============================================================

import ast
import csv
import gc
import json
import math
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

ALIGNED_OUTPUT_DIR = Path(
    "/kaggle/working/aligned_semantic_lookahead_validation"
)
ALIGNED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALIGNED_SELECTIONS_PATH = ALIGNED_OUTPUT_DIR / "aligned_selections.jsonl"
ALIGNED_BRANCHES_PATH = ALIGNED_OUTPUT_DIR / "aligned_branches.jsonl"
ALIGNED_POSITIONS_PATH = ALIGNED_OUTPUT_DIR / "aligned_positions.csv"
ALIGNED_REPORT_PATH = ALIGNED_OUTPUT_DIR / "aligned_report.json"
ALIGNED_ZIP_PATH = Path(
    "/kaggle/working/aligned_semantic_lookahead_validation.zip"
)

BRANCH_BUDGET_ALIGNED = 5


# ------------------------------------------------------------
# Localizar o extraer el checkpoint anterior
# ------------------------------------------------------------

def locate_checkpoint():
    direct = Path("/kaggle/working/frozen_selector_validation")

    required = {
        "baselines.jsonl",
        "lookaheads.jsonl",
        "branches.jsonl",
        "selections.jsonl",
        "validation_report.json",
    }

    if direct.is_dir():
        names = {path.name for path in direct.iterdir()}
        if required.issubset(names):
            return direct

    zip_candidates = []

    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            zip_candidates.extend(
                root.rglob("frozen_selector_validation_checkpoint.zip")
            )

    if not zip_candidates:
        raise FileNotFoundError(
            "No encontré frozen_selector_validation_checkpoint.zip. "
            "Subilo como Kaggle Input o ejecutá primero la validación anterior."
        )

    extract_dir = Path(
        "/kaggle/working/frozen_selector_validation"
    )
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(extract_dir)

    return extract_dir


SOURCE_DIR = locate_checkpoint()

BASELINES_SOURCE = SOURCE_DIR / "baselines.jsonl"
LOOKAHEADS_SOURCE = SOURCE_DIR / "lookaheads.jsonl"
BRANCHES_SOURCE = SOURCE_DIR / "branches.jsonl"
SELECTIONS_SOURCE = SOURCE_DIR / "selections.jsonl"
REPORT_SOURCE = SOURCE_DIR / "validation_report.json"


def read_jsonl(path):
    if not path.exists():
        return []

    with path.open("r", encoding="utf-8") as handle:
        return [
            json.loads(line)
            for line in handle
            if line.strip()
        ]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


# ------------------------------------------------------------
# Alineación Levenshtein con reconstrucción de pares
# ------------------------------------------------------------

def levenshtein_alignment(left, right):
    """
    Devuelve:
      - distancia mínima;
      - pares (índice_left, índice_right).

    None representa una inserción o eliminación.
    """

    n = len(left)
    m = len(right)

    matrix = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(n + 1):
        matrix[i][0] = i

    for j in range(m + 1):
        matrix[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            substitution = (
                0 if left[i - 1] == right[j - 1] else 1
            )

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,
                matrix[i][j - 1] + 1,
                matrix[i - 1][j - 1] + substitution,
            )

    alignment = []
    i = n
    j = m

    while i > 0 or j > 0:
        if (
            i > 0
            and j > 0
            and matrix[i][j]
            == matrix[i - 1][j - 1]
            + (left[i - 1] != right[j - 1])
        ):
            alignment.append((i - 1, j - 1))
            i -= 1
            j -= 1

        elif (
            i > 0
            and matrix[i][j] == matrix[i - 1][j] + 1
        ):
            alignment.append((i - 1, None))
            i -= 1

        else:
            alignment.append((None, j - 1))
            j -= 1

    alignment.reverse()

    return matrix[n][m], alignment


# ------------------------------------------------------------
# Peso léxico de una diferencia alineada
# ------------------------------------------------------------

def aligned_difference_weight(left_text, right_text):
    if left_text == right_text:
        return 0.0

    left_class = token_class(left_text)
    right_class = token_class(right_text)
    classes = {left_class, right_class}

    if classes == {"whitespace"}:
        return 0.0

    if classes == {"identifier"}:
        return 0.20

    if "whitespace" in classes:
        return 0.25

    if classes & {"keyword", "operator", "literal"}:
        return 1.0

    return 0.50


def calculate_aligned_features(
    lookahead,
    baseline,
):
    """
    Excluye el token forzado y alinea solamente la continuación.
    """

    baseline_ids = lookahead["baseline_token_ids"][1:]
    branch_ids = lookahead["branch_token_ids"][1:]

    distance, alignment = levenshtein_alignment(
        baseline_ids,
        branch_ids,
    )

    denominator = max(
        len(baseline_ids),
        len(branch_ids),
        1,
    )

    aligned_persistence = distance / denominator

    position = int(lookahead["position"])
    trace = baseline["trace"]

    # Textos correspondientes a los tokens posteriores
    # al token top-1 original.
    baseline_texts = [
        trace[position + index]["token_text"]
        for index in range(1, len(lookahead["baseline_token_ids"]))
    ]

    branch_texts = lookahead["branch_token_texts"][1:]

    weighted_total = 0.0

    for left_index, right_index in alignment:
        left_text = (
            baseline_texts[left_index]
            if left_index is not None
            else ""
        )

        right_text = (
            branch_texts[right_index]
            if right_index is not None
            else ""
        )

        weighted_total += aligned_difference_weight(
            left_text,
            right_text,
        )

    aligned_weighted_divergence = (
        weighted_total / max(len(alignment), 1)
    )

    positional_mismatches = sum(
        left != right
        for left, right in zip(baseline_ids, branch_ids)
    ) + abs(len(baseline_ids) - len(branch_ids))

    return {
        "aligned_edit_distance": int(distance),
        "aligned_persistence": float(aligned_persistence),
        "aligned_weighted_divergence": float(
            aligned_weighted_divergence
        ),
        "alignment_operations": len(alignment),
        "positional_mismatches": int(positional_mismatches),
        "shift_detected": bool(
            distance < positional_mismatches
        ),
    }


# ------------------------------------------------------------
# Percentiles con empates
# ------------------------------------------------------------

def aligned_midrank_percentiles(rows, field):
    groups = defaultdict(list)

    for row in rows:
        groups[float(row[field])].append(row)

    ordered_values = sorted(groups, reverse=True)
    denominator = max(len(rows) - 1, 1)

    cursor = 1
    scores = {}

    for value in ordered_values:
        group = groups[value]

        mean_rank = cursor + (len(group) - 1) / 2
        percentile = (
            1.0 - (mean_rank - 1) / denominator
        )

        for row in group:
            scores[int(row["position"])] = percentile

        cursor += len(group)

    return scores


def add_aligned_semantic_scores(rows):
    components = (
        "aligned_persistence",
        "aligned_weighted_divergence",
        "anchor_score",
    )

    percentiles = {
        field: aligned_midrank_percentiles(rows, field)
        for field in components
    }

    for row in rows:
        position = int(row["position"])

        row["aligned_semantic_score"] = float(
            np.mean([
                percentiles[field][position]
                for field in components
            ])
        )

    return rows


# ------------------------------------------------------------
# Cargar datos y recalcular señales
# ------------------------------------------------------------

baselines = {
    row["task_id"]: row
    for row in read_jsonl(BASELINES_SOURCE)
}

lookaheads = read_jsonl(LOOKAHEADS_SOURCE)

previous_branches = {
    (row["task_id"], int(row["position"])): row
    for row in read_jsonl(BRANCHES_SOURCE)
}

previous_selections = {
    row["task_id"]: row
    for row in read_jsonl(SELECTIONS_SOURCE)
}

with REPORT_SOURCE.open("r", encoding="utf-8") as handle:
    previous_report = json.load(handle)

failure_task_ids = previous_report[
    "baseline_failures_selected"
]

lookaheads_by_task = defaultdict(list)

for lookahead in lookaheads:
    task_id = lookahead["task_id"]

    aligned_features = calculate_aligned_features(
        lookahead,
        baselines[task_id],
    )

    revised = {
        **lookahead,
        **aligned_features,
    }

    lookaheads_by_task[task_id].append(revised)


aligned_selections = {}
all_aligned_rows = []

for task_id in failure_task_ids:
    rows = add_aligned_semantic_scores(
        lookaheads_by_task[task_id]
    )

    ranked = sorted(
        rows,
        key=lambda row: (
            -row["aligned_semantic_score"],
            deterministic_tie_break(
                task_id,
                "aligned_semantic",
                int(row["position"]),
            ),
        ),
    )

    selected_positions = [
        int(row["position"])
        for row in ranked[:BRANCH_BUDGET_ALIGNED]
    ]

    old_positions = previous_selections[
        task_id
    ]["by_selector"]["semantic_lookahead"]

    aligned_selections[task_id] = {
        "task_id": task_id,
        "aligned_semantic_positions": selected_positions,
        "old_semantic_positions": old_positions,
        "overlap": len(
            set(selected_positions) & set(old_positions)
        ),
        "eligible_positions": len(rows),
    }

    selected_set = set(selected_positions)

    for row in rows:
        row["selected_by_aligned_semantic"] = (
            int(row["position"]) in selected_set
        )
        all_aligned_rows.append(row)


# Guardar selecciones recalculadas
ALIGNED_SELECTIONS_PATH.write_text(
    "".join(
        json.dumps(record, ensure_ascii=False) + "\n"
        for record in aligned_selections.values()
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Determinar qué ramas ya existen y cuáles faltan
# ------------------------------------------------------------

aligned_branch_results = {
    (row["task_id"], int(row["position"])): row
    for row in read_jsonl(ALIGNED_BRANCHES_PATH)
}

missing = []

for task_id, selection in aligned_selections.items():
    for position in selection["aligned_semantic_positions"]:
        key = (task_id, position)

        if key in aligned_branch_results:
            continue

        if key in previous_branches:
            old = previous_branches[key]

            reused = {
                **old,
                "selected_by": ["aligned_semantic"],
                "reused_from_previous_validation": True,
            }

            append_jsonl(
                ALIGNED_BRANCHES_PATH,
                reused,
            )
            aligned_branch_results[key] = reused

        else:
            missing.append(key)


print(
    f"Ramas seleccionadas: "
    f"{len(failure_task_ids) * BRANCH_BUDGET_ALIGNED}"
)
print(
    f"Ramas reutilizadas o ya completas: "
    f"{len(aligned_branch_results)}"
)
print(f"Ramas nuevas que faltan: {len(missing)}")


# ------------------------------------------------------------
# Generar solamente las ramas faltantes
# ------------------------------------------------------------

if missing:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Activá una GPU en Kaggle para generar "
            "las ramas faltantes."
        )

    all_tasks = load_tasks("", num_tasks=164)
    tasks_by_id = {
        task.task_id: task
        for task in all_tasks
    }

    gc.collect()
    torch.cuda.empty_cache()

    print(f"Cargando {VALIDATION_MODEL}...")

    tokenizer = AutoTokenizer.from_pretrained(
        VALIDATION_MODEL
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        VALIDATION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    rows_by_key = {
        (row["task_id"], int(row["position"])): row
        for row in all_aligned_rows
    }

    for index, (task_id, position) in enumerate(
        missing,
        start=1,
    ):
        task = tasks_by_id[task_id]
        baseline = baselines[task_id]
        lookahead = rows_by_key[(task_id, position)]

        prompt = build_prompt(task, tokenizer)

        generated = generate_full_forced_branch(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            prefix_ids=baseline["token_ids"][:position],
            forced_token_id=int(
                lookahead["top2_token_id"]
            ),
        )

        passed, error = evaluate(
            task,
            generated["code"],
            TEST_TIMEOUT_S,
        )

        record = {
            "task_id": task_id,
            "position": position,
            "selected_by": ["aligned_semantic"],
            "passed": bool(passed),
            "error": error,
            "top1_token": lookahead["top1_token"],
            "top2_token": lookahead["top2_token"],
            "top1_token_id": lookahead["top1_token_id"],
            "top2_token_id": lookahead["top2_token_id"],
            "aligned_semantic_score": lookahead[
                "aligned_semantic_score"
            ],
            "aligned_persistence": lookahead[
                "aligned_persistence"
            ],
            "aligned_weighted_divergence": lookahead[
                "aligned_weighted_divergence"
            ],
            "anchor_score": lookahead["anchor_score"],
            "shift_detected": lookahead["shift_detected"],
            "reused_from_previous_validation": False,
            **generated,
        }

        append_jsonl(ALIGNED_BRANCHES_PATH, record)
        aligned_branch_results[(task_id, position)] = record

        print(
            f"[{index}/{len(missing)}] "
            f"{task_id} t={position}: "
            f"{'PASS' if passed else 'FAIL'}"
        )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Resultado final
# ------------------------------------------------------------

task_success = {}

for task_id, selection in aligned_selections.items():
    task_success[task_id] = any(
        aligned_branch_results[
            (task_id, position)
        ]["passed"]
        for position in selection[
            "aligned_semantic_positions"
        ]
    )

recovered_tasks = sum(task_success.values())
total_tasks = len(task_success)
recovery_rate = recovered_tasks / total_tasks

# Intervalo de Wilson 95 %
z = 1.959963984540054
denominator = 1 + z**2 / total_tasks
center = (
    recovery_rate
    + z**2 / (2 * total_tasks)
) / denominator
half_width = (
    z
    * math.sqrt(
        recovery_rate * (1 - recovery_rate) / total_tasks
        + z**2 / (4 * total_tasks**2)
    )
    / denominator
)

shift_rows = [
    row for row in all_aligned_rows
    if row["shift_detected"]
]

selected_rows = [
    row for row in all_aligned_rows
    if row["selected_by_aligned_semantic"]
]

selected_shift_rows = [
    row for row in selected_rows
    if row["shift_detected"]
]

report = {
    "status": "complete",
    "model": VALIDATION_MODEL,
    "tasks": total_tasks,
    "branch_budget_per_task": BRANCH_BUDGET_ALIGNED,
    "lookahead_tokens_total": LOOKAHEAD_TOKENS,
    "continuation_tokens_aligned": LOOKAHEAD_TOKENS - 1,
    "alignment": (
        "unit-cost Levenshtein over continuation tokens; "
        "forced token excluded"
    ),
    "semantic_components": [
        "aligned_persistence",
        "aligned_weighted_divergence",
        "anchor_score",
    ],
    "recovered_tasks": recovered_tasks,
    "recovery_rate": recovery_rate,
    "wilson_95_interval": [
        center - half_width,
        center + half_width,
    ],
    "total_lookahead_positions": len(all_aligned_rows),
    "shift_affected_positions": len(shift_rows),
    "shift_affected_rate": (
        len(shift_rows) / len(all_aligned_rows)
    ),
    "selected_shift_affected_positions": len(
        selected_shift_rows
    ),
    "mean_top5_overlap_with_old_selector": float(
        np.mean([
            selection["overlap"]
            for selection in aligned_selections.values()
        ])
    ),
    "task_results": task_success,
}

ALIGNED_REPORT_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# CSV de posiciones y métricas
csv_fields = [
    "task_id",
    "position",
    "top1_token",
    "top2_token",
    "aligned_persistence",
    "aligned_weighted_divergence",
    "anchor_score",
    "aligned_semantic_score",
    "shift_detected",
    "selected_by_aligned_semantic",
    "persistence_after_forced",
    "weighted_semantic_divergence",
    "normalized_edit_distance",
    "baseline_snippet",
    "branch_snippet",
]

with ALIGNED_POSITIONS_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=csv_fields,
    )
    writer.writeheader()

    for row in sorted(
        all_aligned_rows,
        key=lambda item: (
            item["task_id"],
            int(item["position"]),
        ),
    ):
        writer.writerow({
            field: row.get(field)
            for field in csv_fields
        })


# ZIP descargable
if ALIGNED_ZIP_PATH.exists():
    ALIGNED_ZIP_PATH.unlink()

with zipfile.ZipFile(
    ALIGNED_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in ALIGNED_OUTPUT_DIR.iterdir():
        if path.is_file():
            archive.write(path, arcname=path.name)


print("\nRESULTADO FINAL")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nDescargá este archivo:")
print(ALIGNED_ZIP_PATH)

Ramas seleccionadas: 100
Ramas reutilizadas o ya completas: 100
Ramas nuevas que faltan: 0

RESULTADO FINAL
{
  "status": "complete",
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "tasks": 20,
  "branch_budget_per_task": 5,
  "lookahead_tokens_total": 12,
  "continuation_tokens_aligned": 11,
  "alignment": "unit-cost Levenshtein over continuation tokens; forced token excluded",
  "semantic_components": [
    "aligned_persistence",
    "aligned_weighted_divergence",
    "anchor_score"
  ],
  "recovered_tasks": 10,
  "recovery_rate": 0.5,
  "wilson_95_interval": [
    0.2992980081982123,
    0.7007019918017877
  ],
  "total_lookahead_positions": 1649,
  "shift_affected_positions": 679,
  "shift_affected_rate": 0.4117647058823529,
  "selected_shift_affected_positions": 15,
  "mean_top5_overlap_with_old_selector": 3.35,
  "task_results": {
    "HumanEval/140": true,
    "HumanEval/93": false,
    "HumanEval/10": false,
    "HumanEval/77": true,
    "HumanEval/141": true,
    "HumanEval/15